# 02 — Recalibrate Simple Signal Grid  
## Locked 2621 bucketed Core / Secondary baseline

This notebook is the cleaned baseline for the SPX VRP simple signal grid.

### Current locked candidate

`locked_2621_win_band_25bps_conditional`

### What this notebook does

1. Loads the production feature panel.
2. Loads the completed naked ATM put outcome panel.
3. Builds the complete-date matrix representation.
4. Evaluates the locked bucketed **Core / Secondary 2621** candidate.
5. Uses the final blended tenor priority:
   - Core first.
   - Secondary only if no Core tenor qualifies.
   - Within the active layer, rank by layer-specific 2621-conditional win probability.
   - Treat tenors within 25 bps of the best conditional win probability as near-ties.
   - Use conditional average P&L/day as the tie-break inside that near-tie set.
6. Saves selected trades, summary tables, thresholds, tenor-priority metrics, and worst-trade audits.

### What was intentionally removed

The executable path no longer includes:

- broad grid sweeps
- pair 144 intermediate searches
- old P&L/day-only tenor priority logic
- conditional priority bakeoff code
- 2026 stress-diagnostic exploratory code
- line-fitting / smoothing experiments

Those steps were useful research, but this file is now meant to be a stable rerunnable baseline.


In [1]:
# Cell 1 — Setup, imports, paths, and project metadata

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
# This notebook is intended to run from either the project root or a
# notebooks/ subfolder. If neither location has a data/ directory, it falls
# back to the project path used during development.
# ---------------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
AUDIT_DIR = DATA_DIR / "audit"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_PANEL_PATH = PROCESSED_DIR / "production_feature_panel_v0_1.parquet"
OUTCOME_PATH = PROCESSED_DIR / "naked_atm_put_eod_panel_v0_1.parquet"

# ---------------------------------------------------------------------
# Tenor universe and grouping
# ---------------------------------------------------------------------

PRODUCTION_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

GROUP_TENORS = {
    "front": [9, 12, 15],
    "middle": [18, 21, 24],
    "back": [27, 30, 33],
}

TENOR_GROUP_MAP = {
    tenor: group
    for group, tenors in GROUP_TENORS.items()
    for tenor in tenors
}

# ---------------------------------------------------------------------
# Locked model metadata
# ---------------------------------------------------------------------
# Current preferred research candidate:
#   locked_2621_win_band_25bps_conditional
#
# Important conventions:
#   - Thresholds are bucketed front / middle / back, not line-fitted.
#   - Core is checked before Secondary.
#   - Secondary is checked only when no Core tenor qualifies.
#   - Within the active layer, tenor priority is based on layer-specific
#     2621-conditional win probability.
#   - Tenors within 25 bps of the best conditional win probability are treated
#     as near-ties; conditional average P&L/day chooses among that near-tie set.
#   - Raw average P&L is not used for tenor priority.
# ---------------------------------------------------------------------

MODEL_LABEL = "locked_2621_win_band_25bps_conditional"
WIN_BAND = 0.0025

print("Current working directory:", cwd)
print("Project root:", PROJECT_ROOT)
print("Feature panel path:", FEATURE_PANEL_PATH)
print("Outcome path:", OUTCOME_PATH)
print("Feature panel exists:", FEATURE_PANEL_PATH.exists())
print("Outcome file exists:", OUTCOME_PATH.exists())
print("Model label:", MODEL_LABEL)
print("Win band:", WIN_BAND)


Current working directory: C:\Users\patri\vrp_project\notebooks v1
Project root: C:\Users\patri\vrp_project
Feature panel path: C:\Users\patri\vrp_project\data\processed\production_feature_panel_v0_1.parquet
Outcome path: C:\Users\patri\vrp_project\data\processed\naked_atm_put_eod_panel_v0_1.parquet
Feature panel exists: True
Outcome file exists: True
Model label: locked_2621_win_band_25bps_conditional
Win band: 0.0025


In [2]:
# Cell 2 — Small reusable data-cleaning helpers

def get_col(df: pd.DataFrame, candidates, required=True, label=None):
    """
    Return the first matching column from a candidate list.

    Matching is case-insensitive, but the original DataFrame column name is
    returned. This keeps the notebook robust to small naming changes in source
    files.
    """
    lower_map = {str(c).lower(): c for c in df.columns}

    for c in candidates:
        if c in df.columns:
            return c

        c_lower = str(c).lower()
        if c_lower in lower_map:
            return lower_map[c_lower]

    if required:
        raise KeyError(
            f"Missing column for {label or candidates}. "
            f"Tried: {candidates}. Available columns: {list(df.columns)}"
        )

    return None


def parse_project_date_series(s: pd.Series) -> pd.Series:
    """
    Parse project date columns safely.

    Important:
      Numeric yyyymmdd values such as 20190805 must be parsed as calendar
      dates, not as nanoseconds since 1970.
    """
    s = s.copy()

    if pd.api.types.is_datetime64_any_dtype(s):
        return pd.to_datetime(s)

    s_str = s.astype("string").str.strip()
    s_str = s_str.str.replace(r"\.0$", "", regex=True)

    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    is_yyyymmdd = s_str.str.fullmatch(r"\d{8}", na=False)

    if is_yyyymmdd.any():
        parsed.loc[is_yyyymmdd] = pd.to_datetime(
            s_str.loc[is_yyyymmdd],
            format="%Y%m%d",
            errors="coerce",
        )

    remaining = ~is_yyyymmdd

    if remaining.any():
        parsed.loc[remaining] = pd.to_datetime(
            s_str.loc[remaining],
            errors="coerce",
        )

    return parsed


def clean_binary_series(s: pd.Series) -> pd.Series:
    """
    Convert bool / numeric / string binary values into float 0/1.
    Missing or unresolved values stay NaN.
    """
    if pd.api.types.is_bool_dtype(s):
        return s.astype(float)

    numeric = pd.to_numeric(s, errors="coerce")

    if numeric.notna().mean() > 0.95:
        return numeric.astype(float)

    return (
        s.astype("string")
        .str.lower()
        .str.strip()
        .map({
            "true": 1.0,
            "false": 0.0,
            "yes": 1.0,
            "no": 0.0,
            "1": 1.0,
            "0": 0.0,
            "1.0": 1.0,
            "0.0": 0.0,
        })
    )


def require_columns(df: pd.DataFrame, required_cols, label="DataFrame"):
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"{label} is missing required columns: {missing_cols}")

In [3]:
# Cell 3 — Load and validate the production feature panel

if not FEATURE_PANEL_PATH.exists():
    raise FileNotFoundError(f"Missing feature panel: {FEATURE_PANEL_PATH}")

features = pd.read_parquet(FEATURE_PANEL_PATH)
features["date"] = pd.to_datetime(features["date"]).dt.normalize()
features = features.sort_values(["date", "tenor"]).reset_index(drop=True)

required_feature_cols = [
    "date",
    "tenor",
    "tenor_group",
    "spx_close",
    "implied_variance",
    "forecast_variance",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
]

require_columns(features, required_feature_cols, label="Feature panel")

features = features[features["tenor"].isin(PRODUCTION_TENORS)].copy()
features["tenor"] = features["tenor"].astype(int)

eligible = features.dropna(
    subset=[
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "rv21d",
        "rsi14",
        "implied_variance",
        "forecast_variance",
    ]
).copy()

eligible = eligible.sort_values(["date", "tenor"]).reset_index(drop=True)

dupes = eligible.duplicated(subset=["date", "tenor"]).sum()

print("Loaded feature panel")
print("  Shape:", features.shape)
print("  Date range:", features["date"].min(), "to", features["date"].max())
print("  Tenors:", sorted(features["tenor"].dropna().unique().tolist()))

print("\nEligible feature panel")
print("  Shape:", eligible.shape)
print("  Date range:", eligible["date"].min(), "to", eligible["date"].max())
print("  Tenors:", sorted(eligible["tenor"].dropna().unique().tolist()))
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        eligible[
            eligible.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Duplicate date/tenor rows found in eligible feature panel.")

# rv21d and rsi14 are market-level filters, so they should be identical
# across all tenors on a given date.
market_filter_counts = (
    eligible
    .groupby("date")[["rv21d", "rsi14"]]
    .nunique(dropna=False)
)

bad_market_filter_dates = market_filter_counts[
    (market_filter_counts["rv21d"] > 1) |
    (market_filter_counts["rsi14"] > 1)
]

print("  Dates with inconsistent rv21d/rsi14 across tenors:", len(bad_market_filter_dates))

if len(bad_market_filter_dates) > 0:
    display(bad_market_filter_dates.head(20))
    raise ValueError("rv21d or rsi14 differs across tenors on the same date.")

display(eligible.head(10))

Loaded feature panel
  Shape: (18099, 11)
  Date range: 2018-06-25 00:00:00 to 2026-06-25 00:00:00
  Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]

Eligible feature panel
  Shape: (15750, 11)
  Date range: 2019-07-10 00:00:00 to 2026-06-25 00:00:00
  Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
  Duplicate date/tenor rows: 0
  Dates with inconsistent rv21d/rsi14 across tenors: 0


,date,tenor,tenor_group,spx_close,implied_variance,forecast_variance,vrp_log,vrp_z_3m,vrp_z_1y,rv21d,rsi14
0,2019-07-10,9,front,"2,993.070000",0.010209,0.010971,-0.071968,-1.465831,-0.563268,7.712163,67.404430
1,2019-07-10,12,front,"2,993.070000",0.011378,0.009403,0.190606,-1.095112,-0.216897,7.712163,67.404430
2,2019-07-10,15,front,"2,993.070000",0.012079,0.010344,0.155121,-1.128973,-0.216182,7.712163,67.404430
3,2019-07-10,18,middle,"2,993.070000",0.013868,0.009403,0.388501,-0.584906,0.141210,7.712163,67.404430
4,2019-07-10,21,middle,"2,993.070000",0.015711,0.010075,0.444317,-0.401009,0.274458,7.712163,67.404430
5,2019-07-10,24,middle,"2,993.070000",0.016752,0.009991,0.516811,-0.241134,0.377281,7.712163,67.404430
6,2019-07-10,27,back,"2,993.070000",0.016953,0.009926,0.535301,-0.246532,0.381816,7.712163,67.404430
7,2019-07-10,30,back,"2,993.070000",0.017114,0.010344,0.503503,-0.342138,0.323409,7.712163,67.404430
8,2019-07-10,33,back,"2,993.070000",0.017323,0.009831,0.566508,-0.216783,0.392516,7.712163,67.404430
9,2019-07-11,9,front,"2,999.910000",0.010349,0.008859,0.155524,-0.889726,-0.180556,7.676858,68.576313


In [4]:
# Cell 4 — Load naked ATM put outcomes and build the completed research panel

if not OUTCOME_PATH.exists():
    raise FileNotFoundError(f"Missing outcome file: {OUTCOME_PATH}")

outcomes_raw = pd.read_parquet(OUTCOME_PATH)

# ---------------------------------------------------------------------
# Column mapping
# ---------------------------------------------------------------------

date_col = get_col(
    outcomes_raw,
    ["trade_dt", "trade_date", "date", "timestamp", "datetime"],
    label="date",
)

tenor_col = get_col(
    outcomes_raw,
    ["entry_tenor", "tenor", "target_tenor", "actual_dte", "dte"],
    label="tenor",
)

win_col = get_col(
    outcomes_raw,
    ["win_clean", "win", "test_win"],
    label="win",
)

expired_otm_col = get_col(
    outcomes_raw,
    ["expired_otm_clean", "expired_otm"],
    required=False,
    label="expired_otm",
)

pnl_dollars_col = get_col(
    outcomes_raw,
    ["normalized_pnl_dollars", "test_pnl", "sized_pnl"],
    required=False,
    label="normalized_pnl_dollars",
)

pnl_pct_col = get_col(
    outcomes_raw,
    ["normalized_pnl_pct"],
    required=False,
    label="normalized_pnl_pct",
)

actual_dte_col = get_col(
    outcomes_raw,
    ["actual_dte"],
    required=False,
    label="actual_dte",
)

expiry_spx_col = get_col(
    outcomes_raw,
    ["expiry_spx_close"],
    required=False,
    label="expiry_spx_close",
)

entry_credit_col = get_col(
    outcomes_raw,
    ["entry_credit_points", "atm_put_mid"],
    required=False,
    label="entry_credit_points",
)

short_pnl_points_col = get_col(
    outcomes_raw,
    ["short_option_pnl_points"],
    required=False,
    label="short_option_pnl_points",
)

print("Outcome column mapping:")
print({
    "date_col": date_col,
    "tenor_col": tenor_col,
    "win_col": win_col,
    "expired_otm_col": expired_otm_col,
    "pnl_dollars_col": pnl_dollars_col,
    "pnl_pct_col": pnl_pct_col,
    "actual_dte_col": actual_dte_col,
    "expiry_spx_col": expiry_spx_col,
    "entry_credit_col": entry_credit_col,
    "short_pnl_points_col": short_pnl_points_col,
})

# ---------------------------------------------------------------------
# Standardize outcomes
# ---------------------------------------------------------------------

outcomes = pd.DataFrame({
    "date": parse_project_date_series(outcomes_raw[date_col]).dt.normalize(),
    "tenor": pd.to_numeric(outcomes_raw[tenor_col], errors="coerce").astype("Int64"),
    "win_clean": clean_binary_series(outcomes_raw[win_col]),
})

if expired_otm_col is not None:
    outcomes["expired_otm_clean"] = clean_binary_series(outcomes_raw[expired_otm_col])
else:
    outcomes["expired_otm_clean"] = np.nan

if pnl_dollars_col is not None:
    outcomes["normalized_pnl_dollars"] = pd.to_numeric(outcomes_raw[pnl_dollars_col], errors="coerce")
else:
    outcomes["normalized_pnl_dollars"] = np.nan

if pnl_pct_col is not None:
    outcomes["normalized_pnl_pct"] = pd.to_numeric(outcomes_raw[pnl_pct_col], errors="coerce")
else:
    outcomes["normalized_pnl_pct"] = np.nan

if actual_dte_col is not None:
    outcomes["actual_dte"] = pd.to_numeric(outcomes_raw[actual_dte_col], errors="coerce")
else:
    outcomes["actual_dte"] = np.nan

if expiry_spx_col is not None:
    outcomes["expiry_spx_close"] = pd.to_numeric(outcomes_raw[expiry_spx_col], errors="coerce")
else:
    outcomes["expiry_spx_close"] = np.nan

if entry_credit_col is not None:
    outcomes["entry_credit_points"] = pd.to_numeric(outcomes_raw[entry_credit_col], errors="coerce")
else:
    outcomes["entry_credit_points"] = np.nan

if short_pnl_points_col is not None:
    outcomes["short_option_pnl_points"] = pd.to_numeric(outcomes_raw[short_pnl_points_col], errors="coerce")
else:
    outcomes["short_option_pnl_points"] = np.nan

outcomes = outcomes[outcomes["tenor"].isin(PRODUCTION_TENORS)].copy()
outcomes = outcomes.dropna(subset=["date", "tenor"]).copy()
outcomes["tenor"] = outcomes["tenor"].astype(int)

dupes = outcomes.duplicated(subset=["date", "tenor"]).sum()

print("\nStandardized outcome panel")
print("  Shape:", outcomes.shape)
print("  Date range:", outcomes["date"].min(), "to", outcomes["date"].max())
print("  Tenors:", sorted(outcomes["tenor"].dropna().unique().tolist()))
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        outcomes[
            outcomes.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Outcome panel has duplicate date/tenor rows.")

# ---------------------------------------------------------------------
# Join outcomes to eligible features.
# ---------------------------------------------------------------------
#
# Missing outcomes near the right edge are expected because later-dated
# longer-tenor trades may not have expired yet. Those rows are dropped.
#
# Missing outcomes before the last completed outcome date for that tenor are
# treated as a data hole and raise an error.
# ---------------------------------------------------------------------

research_panel_raw = eligible.merge(
    outcomes,
    on=["date", "tenor"],
    how="left",
)

completed_outcomes = outcomes.dropna(subset=["win_clean"]).copy()

last_completed_date_by_tenor = (
    completed_outcomes
    .groupby("tenor")["date"]
    .max()
    .rename("last_completed_outcome_date")
    .reset_index()
)

missing_outcomes = (
    research_panel_raw[research_panel_raw["win_clean"].isna()]
    [["date", "tenor", "tenor_group", "spx_close", "vrp_log", "vrp_z_3m", "vrp_z_1y", "rv21d", "rsi14"]]
    .copy()
)

missing_outcomes = missing_outcomes.merge(
    last_completed_date_by_tenor,
    on="tenor",
    how="left",
)

unexpected_missing = missing_outcomes[
    missing_outcomes["date"] <= missing_outcomes["last_completed_outcome_date"]
].copy()

print("\nResearch panel after outcome join")
print("  Shape:", research_panel_raw.shape)
print("  Rows with missing win_clean:", int(research_panel_raw["win_clean"].isna().sum()))

print("\nLast completed outcome date by tenor:")
display(last_completed_date_by_tenor)

if len(missing_outcomes) > 0:
    missing_summary = (
        missing_outcomes
        .groupby("tenor")
        .agg(
            missing_rows=("date", "size"),
            first_missing_date=("date", "min"),
            last_missing_date=("date", "max"),
            last_completed_outcome_date=("last_completed_outcome_date", "max"),
        )
        .reset_index()
    )

    print("\nMissing outcome summary by tenor:")
    display(missing_summary)

if len(unexpected_missing) > 0:
    print("\nUnexpected missing outcomes found before/equal to last completed date for that tenor:")
    display(unexpected_missing.head(30))
    raise ValueError("Unexpected non-right-edge missing outcomes found.")

print("\nRight-edge censored rows to drop:", len(missing_outcomes))

research_panel = research_panel_raw.dropna(subset=["win_clean"]).copy()
research_panel = research_panel.sort_values(["date", "tenor"]).reset_index(drop=True)

required_outcome_cols = ["win_clean", "normalized_pnl_dollars", "actual_dte"]
require_columns(research_panel, required_outcome_cols, label="Completed research panel")

if research_panel[required_outcome_cols].isna().any().any():
    display(research_panel[research_panel[required_outcome_cols].isna().any(axis=1)].head(20))
    raise ValueError("Completed research panel has missing win/P&L/actual_dte values.")

print("\nFinal completed research panel")
print("  Shape:", research_panel.shape)
print("  Date range:", research_panel["date"].min(), "to", research_panel["date"].max())
print("  Tenors:", sorted(research_panel["tenor"].dropna().unique().tolist()))
print("  Overall win rate:", research_panel["win_clean"].mean())

outcome_by_tenor = (
    research_panel
    .groupby("tenor")
    .agg(
        rows=("win_clean", "size"),
        win_rate=("win_clean", "mean"),
        avg_pnl_dollars=("normalized_pnl_dollars", "mean"),
        avg_pnl_pct=("normalized_pnl_pct", "mean"),
        avg_actual_dte=("actual_dte", "mean"),
    )
    .reset_index()
)

print("\nOutcome summary by tenor:")
display(outcome_by_tenor)

Outcome column mapping:
{'date_col': 'trade_dt', 'tenor_col': 'entry_tenor', 'win_col': 'win', 'expired_otm_col': 'expired_otm', 'pnl_dollars_col': 'normalized_pnl_dollars', 'pnl_pct_col': None, 'actual_dte_col': 'actual_dte', 'expiry_spx_col': 'expiry_spx_close', 'entry_credit_col': 'entry_credit_points', 'short_pnl_points_col': 'short_option_pnl_points'}

Standardized outcome panel
  Shape: (18099, 10)
  Date range: 2018-06-25 00:00:00 to 2026-06-25 00:00:00
  Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
  Duplicate date/tenor rows: 0

Research panel after outcome join
  Shape: (15750, 19)
  Rows with missing win_clean: 153

Last completed outcome date by tenor:


,tenor,last_completed_outcome_date
0,9,2026-06-12
1,12,2026-06-09
2,15,2026-06-05
3,18,2026-06-03
4,21,2026-05-29
5,24,2026-05-28
6,27,2026-05-22
7,30,2026-05-22
8,33,2026-05-19



Missing outcome summary by tenor:


,tenor,missing_rows,first_missing_date,last_missing_date,last_completed_outcome_date
0,9,8,2026-06-15,2026-06-25,2026-06-12
1,12,11,2026-06-10,2026-06-25,2026-06-09
2,15,13,2026-06-08,2026-06-25,2026-06-05
3,18,15,2026-06-04,2026-06-25,2026-06-03
4,21,18,2026-06-01,2026-06-25,2026-05-29
5,24,19,2026-05-29,2026-06-25,2026-05-28
6,27,22,2026-05-26,2026-06-25,2026-05-22
7,30,22,2026-05-26,2026-06-25,2026-05-22
8,33,25,2026-05-20,2026-06-25,2026-05-19



Right-edge censored rows to drop: 153

Final completed research panel
  Shape: (15597, 19)
  Date range: 2019-07-10 00:00:00 to 2026-06-12 00:00:00
  Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
  Overall win rate: 0.7633519266525614

Outcome summary by tenor:


,tenor,rows,win_rate,avg_pnl_dollars,avg_pnl_pct,avg_actual_dte
0,9,1742,0.731343,"1,123.741635",NaN,8.940299
1,12,1739,0.742381,"1,816.837565",NaN,11.848189
2,15,1737,0.748417,"2,462.510510",NaN,15.940127
3,18,1735,0.753890,"2,648.826727",NaN,17.438040
4,21,1732,0.760393,"3,264.124558",NaN,21.727483
5,24,1731,0.767187,"3,535.662513",NaN,23.038128
6,27,1728,0.785301,"4,430.317093",NaN,27.284722
7,30,1728,0.788773,"4,900.713063",NaN,29.941551
8,33,1725,0.793043,"5,573.755492",NaN,32.848116


In [5]:
# Cell 5 — Build complete-date sweep panel and matrix representation

# ---------------------------------------------------------------------
# Outcome normalization
# ---------------------------------------------------------------------
# The selected-tenor rule uses conditional P&L/day as a tie-breaker. This is
# not the same as using raw average P&L, which mechanically favors longer DTE.
# ---------------------------------------------------------------------

research_panel = research_panel.copy()

with np.errstate(divide="ignore", invalid="ignore"):
    research_panel["outcome_pnl_per_day"] = (
        research_panel["normalized_pnl_dollars"] / research_panel["actual_dte"]
    )

research_panel.loc[
    ~np.isfinite(research_panel["outcome_pnl_per_day"]),
    "outcome_pnl_per_day",
] = np.nan

# ---------------------------------------------------------------------
# Sweep panel cleaning
# ---------------------------------------------------------------------
# Keep only the columns needed for the locked signal evaluation and future
# overlay/sizing work.
# ---------------------------------------------------------------------

sweep_panel = research_panel[
    [
        "date",
        "tenor",
        "tenor_group",
        "spx_close",
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "rv21d",
        "rsi14",
        "win_clean",
        "normalized_pnl_dollars",
        "actual_dte",
        "outcome_pnl_per_day",
    ]
].copy()

numeric_cols = [
    "tenor",
    "spx_close",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
    "win_clean",
    "normalized_pnl_dollars",
    "actual_dte",
    "outcome_pnl_per_day",
]

for col in numeric_cols:
    sweep_panel[col] = pd.to_numeric(sweep_panel[col], errors="coerce")

before_drop = len(sweep_panel)

sweep_panel = sweep_panel.dropna(
    subset=[
        "date",
        "tenor",
        "tenor_group",
        "spx_close",
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "rv21d",
        "rsi14",
        "win_clean",
        "normalized_pnl_dollars",
        "actual_dte",
        "outcome_pnl_per_day",
    ]
).copy()

after_drop = len(sweep_panel)

print("\nSweep panel row cleaning")
print("  Rows before drop:", before_drop)
print("  Rows after drop: ", after_drop)
print("  Rows dropped:     ", before_drop - after_drop)

dupes = sweep_panel.duplicated(subset=["date", "tenor"]).sum()
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        sweep_panel[
            sweep_panel.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Duplicate date/tenor rows in sweep_panel.")

# ---------------------------------------------------------------------
# Complete-date filter
# ---------------------------------------------------------------------
# Keep only dates where all 9 tenors have completed outcomes. This keeps trade
# frequency comparisons clean and prevents right-edge censoring from biasing
# tenor selection.
# ---------------------------------------------------------------------

REQUIRED_TENOR_COUNT = len(PRODUCTION_TENORS)

tenor_count_by_date = (
    sweep_panel
    .groupby("date")["tenor"]
    .nunique()
    .rename("tenor_count")
)

complete_dates = tenor_count_by_date[tenor_count_by_date == REQUIRED_TENOR_COUNT].index
incomplete_dates = tenor_count_by_date[tenor_count_by_date != REQUIRED_TENOR_COUNT]

print("\nComplete-date filter")
print("  Original sweep rows:", len(sweep_panel))
print("  Original sweep dates:", sweep_panel["date"].nunique())
print("  Complete dates:", len(complete_dates))
print("  Incomplete dates:", len(incomplete_dates))

if len(incomplete_dates) > 0:
    print("\nIncomplete date summary:")
    display(
        incomplete_dates
        .reset_index()
        .sort_values("date")
    )

sweep_panel = sweep_panel[sweep_panel["date"].isin(complete_dates)].copy()
sweep_panel = sweep_panel.sort_values(["date", "tenor"]).reset_index(drop=True)

sweep_dates = pd.Index(sorted(sweep_panel["date"].unique()))
num_sweep_dates = len(sweep_dates)
TRADE_FREQUENCY_DENOMINATOR = num_sweep_dates

print("\nClean sweep panel")
print("  Rows:", len(sweep_panel))
print("  Dates:", num_sweep_dates)
print("  Date range:", sweep_dates.min(), "to", sweep_dates.max())
print("  Trade frequency denominator:", TRADE_FREQUENCY_DENOMINATOR)

clean_tenor_count_by_date = sweep_panel.groupby("date")["tenor"].nunique()

if not (clean_tenor_count_by_date == REQUIRED_TENOR_COUNT).all():
    raise ValueError("Some remaining dates do not have all 9 tenors.")

print("\nRows by tenor group after complete-date filter:")
display(
    sweep_panel
    .groupby("tenor_group")
    .agg(
        rows=("win_clean", "size"),
        win_rate=("win_clean", "mean"),
        avg_vrp=("vrp_log", "mean"),
        avg_z3m=("vrp_z_3m", "mean"),
        avg_z1y=("vrp_z_1y", "mean"),
        avg_rv21d=("rv21d", "mean"),
        avg_rsi14=("rsi14", "mean"),
        avg_pnl_per_day=("outcome_pnl_per_day", "mean"),
        total_pnl_dollars=("normalized_pnl_dollars", "sum"),
        total_actual_dte=("actual_dte", "sum"),
    )
    .assign(
        aggregate_pnl_per_day=lambda x: x["total_pnl_dollars"] / x["total_actual_dte"]
    )
    .reset_index()
)

# ---------------------------------------------------------------------
# Matrix representation
# ---------------------------------------------------------------------
# All strategy logic below uses matrix arrays:
#   rows = dates
#   columns = tenors [9, 12, ..., 33]
# ---------------------------------------------------------------------

TENOR_ARRAY = np.array(PRODUCTION_TENORS, dtype=int)
TENOR_ORDER = TENOR_ARRAY.tolist()
TENOR_TO_COL = {int(t): i for i, t in enumerate(TENOR_ARRAY)}

GROUP_COLS = {
    group: [TENOR_TO_COL[t] for t in tenors]
    for group, tenors in GROUP_TENORS.items()
}


def pivot_to_matrix(value_col):
    mat_df = (
        sweep_panel
        .pivot(index="date", columns="tenor", values=value_col)
        .reindex(index=sweep_dates, columns=PRODUCTION_TENORS)
    )

    if mat_df.isna().any().any():
        bad = mat_df[mat_df.isna().any(axis=1)].head(10)
        display(bad)
        raise ValueError(f"Matrix for {value_col} contains missing values.")

    return mat_df.to_numpy()


spx_mat = pivot_to_matrix("spx_close")
spx_close_mat = spx_mat
vrp_mat = pivot_to_matrix("vrp_log")
z3m_mat = pivot_to_matrix("vrp_z_3m")
z1y_mat = pivot_to_matrix("vrp_z_1y")
win_mat = pivot_to_matrix("win_clean")
pnl_mat = pivot_to_matrix("normalized_pnl_dollars")
actual_dte_mat = pivot_to_matrix("actual_dte")
pnl_per_day_mat = pivot_to_matrix("outcome_pnl_per_day")

rv21d_by_date = (
    sweep_panel.groupby("date")["rv21d"].first().reindex(sweep_dates).to_numpy()
)

rsi_by_date = (
    sweep_panel.groupby("date")["rsi14"].first().reindex(sweep_dates).to_numpy()
)

print("\nMatrix build complete")
print("  Matrix shape:", vrp_mat.shape)
print("  Tenor order:", TENOR_ORDER)



Sweep panel row cleaning
  Rows before drop: 15597
  Rows after drop:  15597
  Rows dropped:      0
  Duplicate date/tenor rows: 0

Complete-date filter
  Original sweep rows: 15597
  Original sweep dates: 1742
  Complete dates: 1725
  Incomplete dates: 17

Incomplete date summary:


,date,tenor_count
0,2026-05-20,8
1,2026-05-21,8
2,2026-05-22,8
3,2026-05-26,6
4,2026-05-27,6
5,2026-05-28,6
6,2026-05-29,5
7,2026-06-01,4
8,2026-06-02,4
9,2026-06-03,4



Clean sweep panel
  Rows: 15525
  Dates: 1725
  Date range: 2019-07-10 00:00:00 to 2026-05-19 00:00:00
  Trade frequency denominator: 1725

Rows by tenor group after complete-date filter:


,tenor_group,rows,win_rate,avg_vrp,avg_z3m,avg_z1y,avg_rv21d,avg_rsi14,avg_pnl_per_day,total_pnl_dollars,total_actual_dte,aggregate_pnl_per_day
0,back,5175,0.788792,0.508307,0.068748,0.076143,16.720316,55.807832,164.798276,"25,653,165.719078",155382,165.097410
1,front,5175,0.743188,0.667672,0.071348,0.087078,16.720316,55.807832,144.656581,"9,471,878.998276",63373,149.462374
2,middle,5175,0.760580,0.524071,0.068264,0.079054,16.720316,55.807832,150.822859,"16,247,873.322091",107302,151.421906



Matrix build complete
  Matrix shape: (1725, 9)
  Tenor order: [9, 12, 15, 18, 21, 24, 27, 30, 33]


## Locked model specification

### Threshold surface

The production research candidate uses **bucketed** front / middle / back thresholds. Do **not** line-fit or smooth these thresholds in this baseline notebook.

| Layer | Parameter | Front | Middle | Back |
|---|---|---:|---:|---:|
| Core | VRP | 0.60 | 0.65 | 0.70 |
| Core | z3m | 0.55 | 0.75 | 0.75 |
| Core | z1y | 0.65 | 0.65 | 0.75 |
| Core | RSI cap | 70 | 68 | 66 |
| Core | RV21D floor | 8.5 | 8.5 | 8.5 |
| Secondary | VRP | 0.60 | 0.60 | 0.70 |
| Secondary | z3m | 0.50 | 0.50 | 0.50 |
| Secondary | z1y | 0.40 | 0.50 | 0.50 |
| Secondary | RSI cap | 74 | 70 | 68 |
| Secondary | RV21D floor | 6.5 | 6.5 | 6.5 |

### Selection convention

1. Evaluate all Core tenors.
2. If any Core tenor qualifies, choose one Core tenor and ignore Secondary.
3. If no Core tenor qualifies, evaluate all Secondary tenors and choose one Secondary tenor if any qualify.
4. Maximum one selected trade per date.
5. Tenor priority is conditional on the locked 2621 qualifying universe, not unconditional tenor statistics.


In [6]:
# Cell — Final locked 2621 blended tenor-priority model
#
# Final model convention:
#   1. Thresholds are locked to 2621.
#   2. Core is checked first.
#   3. Secondary is checked only if no Core tenor qualifies.
#   4. Within the active layer:
#        a. Rank tenors by layer-specific 2621-conditional win probability.
#        b. Keep tenors within 25 bps of the best qualifying tenor's win probability.
#        c. Select the tenor with highest layer-specific 2621-conditional avg P&L/day.
#        d. Tie-break by aggregate P&L/day, then conditional win probability, then longer tenor.
#
# Why:
#   - Avoids raw P&L duration bias.
#   - Avoids unconditional tenor statistics.
#   - Avoids pure P&L/day selection taking over the model.
#   - Keeps win probability as the primary criterion.
#
# Expected tenor behavior from bakeoff:
#   - Same front/middle/back mix as win_only.
#   - Main change: inside the back bucket, 27D can beat 33D when their conditional
#     win probabilities are effectively tied and 27D has better P&L/day.

import numpy as np
import pandas as pd
import json
from pathlib import Path

# ---------------------------------------------------------------------
# Robust setup
# ---------------------------------------------------------------------

if "TENOR_ORDER" in globals():
    TENOR_ARRAY = np.array(TENOR_ORDER, dtype=int)

elif "sweep_panel" in globals() and "tenor" in sweep_panel.columns:
    TENOR_ARRAY = np.array(sorted(sweep_panel["tenor"].dropna().unique()), dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

elif "research_panel" in globals() and "tenor" in research_panel.columns:
    TENOR_ARRAY = np.array(sorted(research_panel["tenor"].dropna().unique()), dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

else:
    TENOR_ARRAY = np.array([9, 12, 15, 18, 21, 24, 27, 30, 33], dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

TENOR_TO_COL = {int(t): i for i, t in enumerate(TENOR_ARRAY)}

required_objects = [
    "num_sweep_dates",
    "sweep_dates",
    "vrp_mat",
    "z3m_mat",
    "z1y_mat",
    "win_mat",
    "pnl_mat",
    "actual_dte_mat",
    "rv21d_by_date",
    "rsi_by_date",
    "TRADE_FREQUENCY_DENOMINATOR",
]

missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise NameError(
        "Missing required objects. Run the data-loading / matrix-construction cells first. "
        f"Missing: {missing_objects}"
    )

if "pnl_per_day_mat" not in globals():
    with np.errstate(divide="ignore", invalid="ignore"):
        pnl_per_day_mat = pnl_mat / actual_dte_mat
    pnl_per_day_mat[~np.isfinite(pnl_per_day_mat)] = np.nan

if "AUDIT_DIR" not in globals():
    AUDIT_DIR = Path("data/audit")
else:
    AUDIT_DIR = Path(AUDIT_DIR)

if "PROCESSED_DIR" not in globals():
    PROCESSED_DIR = Path("data/processed")
else:
    PROCESSED_DIR = Path(PROCESSED_DIR)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MODEL_LABEL = "locked_2621_win_band_25bps_conditional"
WIN_BAND = 0.0025

print("Final locked 2621 blended-priority model")
print("Model label:", MODEL_LABEL)
print("Tenors:", TENOR_ARRAY.tolist())
print("Win band:", WIN_BAND)


# ---------------------------------------------------------------------
# Locked 2621 thresholds
# ---------------------------------------------------------------------

LOCKED_2621_CORE = {
    "front": {
        "tenors": [9, 12, 15],
        "vrp": 0.60,
        "z3m": 0.55,
        "z1y": 0.65,
        "rsi_cap": 70,
        "rv21d_floor": 8.5,
    },
    "middle": {
        "tenors": [18, 21, 24],
        "vrp": 0.65,
        "z3m": 0.75,
        "z1y": 0.65,
        "rsi_cap": 68,
        "rv21d_floor": 8.5,
    },
    "back": {
        "tenors": [27, 30, 33],
        "vrp": 0.70,
        "z3m": 0.75,
        "z1y": 0.75,
        "rsi_cap": 66,
        "rv21d_floor": 8.5,
    },
}

LOCKED_2621_SECONDARY = {
    "front": {
        "tenors": [9, 12, 15],
        "vrp": 0.60,
        "z3m": 0.50,
        "z1y": 0.40,
        "rsi_cap": 74,
        "rv21d_floor": 6.5,
    },
    "middle": {
        "tenors": [18, 21, 24],
        "vrp": 0.60,
        "z3m": 0.50,
        "z1y": 0.50,
        "rsi_cap": 70,
        "rv21d_floor": 6.5,
    },
    "back": {
        "tenors": [27, 30, 33],
        "vrp": 0.70,
        "z3m": 0.50,
        "z1y": 0.50,
        "rsi_cap": 68,
        "rv21d_floor": 6.5,
    },
}


# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def tenor_group_for_tenor(tenor):
    tenor = int(tenor)

    if tenor in [9, 12, 15]:
        return "front"
    elif tenor in [18, 21, 24]:
        return "middle"
    elif tenor in [27, 30, 33]:
        return "back"
    else:
        raise ValueError(f"Unexpected tenor: {tenor}")


def build_threshold_table(threshold_spec, layer):
    rows = []

    for group, params in threshold_spec.items():
        for tenor in params["tenors"]:
            rows.append({
                "layer": layer,
                "tenor_group": group,
                "selected_tenor": int(tenor),
                "vrp_threshold": float(params["vrp"]),
                "z3m_threshold": float(params["z3m"]),
                "z1y_threshold": float(params["z1y"]),
                "rsi_cap": float(params["rsi_cap"]),
                "rv21d_floor": float(params["rv21d_floor"]),
            })

    return pd.DataFrame(rows)


def build_qualifies_matrix(threshold_spec):
    """
    Builds date x tenor boolean qualification matrix for locked 2621 thresholds.
    """
    qualifies = np.zeros((num_sweep_dates, len(TENOR_ARRAY)), dtype=bool)

    for col, tenor in enumerate(TENOR_ARRAY):
        group = tenor_group_for_tenor(tenor)
        params = threshold_spec[group]

        qualifies[:, col] = (
            (vrp_mat[:, col] >= float(params["vrp"])) &
            (z3m_mat[:, col] >= float(params["z3m"])) &
            (z1y_mat[:, col] >= float(params["z1y"])) &
            (rsi_by_date <= float(params["rsi_cap"])) &
            (rv21d_by_date >= float(params["rv21d_floor"]))
        )

    return qualifies


def build_layer_conditional_tenor_metrics(layer_name, qualifies_matrix, date_mask):
    """
    Builds tenor ranking metrics conditional on the relevant 2621 candidate universe.

    Core:
      date_mask = all dates.

    Secondary:
      date_mask = dates where no Core tenor qualifies.
    """
    rows = []

    for col, tenor in enumerate(TENOR_ARRAY):
        candidate_mask = (
            date_mask &
            qualifies_matrix[:, col] &
            np.isfinite(win_mat[:, col]) &
            np.isfinite(pnl_mat[:, col]) &
            np.isfinite(actual_dte_mat[:, col]) &
            np.isfinite(pnl_per_day_mat[:, col])
        )

        candidate_count = int(candidate_mask.sum())

        if candidate_count == 0:
            rows.append({
                "layer": layer_name,
                "selected_col": int(col),
                "selected_tenor": int(tenor),
                "tenor_group": tenor_group_for_tenor(tenor),
                "conditional_win_probability": np.nan,
                "conditional_avg_pnl_per_day": np.nan,
                "conditional_median_pnl_per_day": np.nan,
                "conditional_aggregate_pnl_per_day": np.nan,
                "conditional_avg_raw_pnl": np.nan,
                "conditional_avg_actual_dte": np.nan,
                "conditional_worst_trade_pnl": np.nan,
                "conditional_worst_pnl_per_day": np.nan,
                "candidate_count": 0,
            })
            continue

        rows.append({
            "layer": layer_name,
            "selected_col": int(col),
            "selected_tenor": int(tenor),
            "tenor_group": tenor_group_for_tenor(tenor),

            "conditional_win_probability": float(np.nanmean(win_mat[candidate_mask, col])),
            "conditional_avg_pnl_per_day": float(np.nanmean(pnl_per_day_mat[candidate_mask, col])),
            "conditional_median_pnl_per_day": float(np.nanmedian(pnl_per_day_mat[candidate_mask, col])),
            "conditional_aggregate_pnl_per_day": float(
                np.nansum(pnl_mat[candidate_mask, col]) /
                np.nansum(actual_dte_mat[candidate_mask, col])
            ),
            "conditional_avg_raw_pnl": float(np.nanmean(pnl_mat[candidate_mask, col])),
            "conditional_avg_actual_dte": float(np.nanmean(actual_dte_mat[candidate_mask, col])),
            "conditional_worst_trade_pnl": float(np.nanmin(pnl_mat[candidate_mask, col])),
            "conditional_worst_pnl_per_day": float(np.nanmin(pnl_per_day_mat[candidate_mask, col])),
            "candidate_count": candidate_count,
        })

    return pd.DataFrame(rows)


def blended_priority_order(layer_name, qualifying_cols, win_band=WIN_BAND):
    """
    Final blended priority rule.

    Step 1:
      Find best conditional win probability among qualifying tenors.

    Step 2:
      Keep tenors within win_band of that best conditional win probability.

    Step 3:
      Select highest conditional avg P&L/day.
      Tie-break by aggregate P&L/day, then conditional win probability, then longer tenor.
    """
    if len(qualifying_cols) == 0:
        return pd.DataFrame()

    df = tenor_priority_metrics[
        (tenor_priority_metrics["layer"].eq(layer_name)) &
        (tenor_priority_metrics["selected_col"].isin(qualifying_cols)) &
        (tenor_priority_metrics["candidate_count"] > 0)
    ].copy()

    if df.empty:
        return df

    best_win = df["conditional_win_probability"].max()

    df["best_conditional_win_probability"] = best_win
    df["win_probability_gap_to_best"] = best_win - df["conditional_win_probability"]
    df["inside_win_band"] = df["win_probability_gap_to_best"] <= win_band

    df = df[df["inside_win_band"]].copy()

    df = df.sort_values(
        [
            "conditional_avg_pnl_per_day",
            "conditional_aggregate_pnl_per_day",
            "conditional_win_probability",
            "selected_tenor",
        ],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    df["priority_rank_within_signal"] = np.arange(1, len(df) + 1)

    return df


def select_final_model_path():
    """
    Core-first / Secondary-second final path selection.
    """
    selected_rows = []

    core_selected = np.full(num_sweep_dates, -1, dtype=np.int8)
    secondary_selected = np.full(num_sweep_dates, -1, dtype=np.int8)
    combined_selected = np.full(num_sweep_dates, -1, dtype=np.int8)

    for row in range(num_sweep_dates):
        core_qualifying_cols = np.where(core_qualifies_matrix[row, :])[0].astype(int).tolist()

        if core_qualifying_cols:
            ordered = blended_priority_order(
                layer_name="Core",
                qualifying_cols=core_qualifying_cols,
                win_band=WIN_BAND,
            )

            if not ordered.empty:
                selected_col = int(ordered.iloc[0]["selected_col"])

                core_selected[row] = selected_col
                combined_selected[row] = selected_col

                selected_rows.append({
                    "row_idx": int(row),
                    "date": sweep_dates[row],
                    "layer": "Core",
                    "selected_col": selected_col,
                    "selected_tenor": int(TENOR_ARRAY[selected_col]),
                    "tenor_group": tenor_group_for_tenor(TENOR_ARRAY[selected_col]),
                    "num_qualifying_tenors_in_layer": int(len(core_qualifying_cols)),
                    "qualifying_tenors_in_layer": ",".join(
                        [str(int(TENOR_ARRAY[c])) for c in core_qualifying_cols]
                    ),
                    "selection_rule": "win_band_25bps_then_pnl_day",
                    "win_band": WIN_BAND,
                })

            continue

        secondary_qualifying_cols = np.where(secondary_qualifies_matrix[row, :])[0].astype(int).tolist()

        if secondary_qualifying_cols:
            ordered = blended_priority_order(
                layer_name="Secondary",
                qualifying_cols=secondary_qualifying_cols,
                win_band=WIN_BAND,
            )

            if not ordered.empty:
                selected_col = int(ordered.iloc[0]["selected_col"])

                secondary_selected[row] = selected_col
                combined_selected[row] = selected_col

                selected_rows.append({
                    "row_idx": int(row),
                    "date": sweep_dates[row],
                    "layer": "Secondary",
                    "selected_col": selected_col,
                    "selected_tenor": int(TENOR_ARRAY[selected_col]),
                    "tenor_group": tenor_group_for_tenor(TENOR_ARRAY[selected_col]),
                    "num_qualifying_tenors_in_layer": int(len(secondary_qualifying_cols)),
                    "qualifying_tenors_in_layer": ",".join(
                        [str(int(TENOR_ARRAY[c])) for c in secondary_qualifying_cols]
                    ),
                    "selection_rule": "win_band_25bps_then_pnl_day",
                    "win_band": WIN_BAND,
                })

    selected_path = pd.DataFrame(selected_rows)

    return selected_path, {
        "core_selected": core_selected,
        "secondary_selected": secondary_selected,
        "combined_selected": combined_selected,
    }


def attach_trade_outcomes(selected_path):
    """
    Adds outcomes, features, and conditional ranking metrics to selected path.
    """
    if selected_path.empty:
        return selected_path.copy()

    row_idx = selected_path["row_idx"].to_numpy(dtype=int)
    col_idx = selected_path["selected_col"].to_numpy(dtype=int)

    trades = selected_path.copy()

    trades["candidate"] = MODEL_LABEL

    trades["win"] = win_mat[row_idx, col_idx]
    trades["normalized_pnl_dollars"] = pnl_mat[row_idx, col_idx]
    trades["actual_dte"] = actual_dte_mat[row_idx, col_idx]
    trades["pnl_per_day"] = pnl_per_day_mat[row_idx, col_idx]

    trades["vrp_log"] = vrp_mat[row_idx, col_idx]
    trades["vrp_z_3m"] = z3m_mat[row_idx, col_idx]
    trades["vrp_z_1y"] = z1y_mat[row_idx, col_idx]
    trades["rv21d"] = rv21d_by_date[row_idx]
    trades["rsi14"] = rsi_by_date[row_idx]

    if "spx_close_mat" in globals():
        trades["spx_close"] = spx_close_mat[row_idx, col_idx]
    elif "spx_close_by_date" in globals():
        trades["spx_close"] = np.asarray(spx_close_by_date)[row_idx]
    else:
        trades["spx_close"] = np.nan

    trades = trades.merge(
        tenor_priority_metrics,
        on=["layer", "selected_col", "selected_tenor", "tenor_group"],
        how="left",
    )

    trades = trades.sort_values("date").reset_index(drop=True)

    trades["trade_sequence"] = np.arange(1, len(trades) + 1)
    trades["cum_pnl"] = trades["normalized_pnl_dollars"].cumsum()
    trades["running_max_cum_pnl"] = trades["cum_pnl"].cummax()
    trades["drawdown"] = trades["cum_pnl"] - trades["running_max_cum_pnl"]
    trades["rolling_20_trade_pnl"] = trades["normalized_pnl_dollars"].rolling(20).sum()

    trades["year"] = pd.to_datetime(trades["date"]).dt.year
    trades["month"] = pd.to_datetime(trades["date"]).dt.to_period("M").astype(str)

    return trades


def summarize_trade_df(df, label):
    """
    Summary for one path or subgroup.
    """
    trade_count = int(len(df))

    if trade_count == 0:
        return pd.Series({
            "label": label,
            "trade_count": 0,
            "trade_frequency": 0.0,
            "win_rate": np.nan,
            "avg_pnl_per_day": np.nan,
            "median_pnl_per_day": np.nan,
            "aggregate_pnl_per_day": np.nan,
            "total_pnl": 0.0,
            "total_actual_dte": 0.0,
            "worst_trade_pnl": np.nan,
            "worst_pnl_per_day": np.nan,
            "max_drawdown": np.nan,
            "worst_20_trade_sum": np.nan,
            "avg_selected_tenor": np.nan,
        })

    df = df.sort_values("date").copy()

    df["local_cum_pnl"] = df["normalized_pnl_dollars"].cumsum()
    df["local_running_max"] = df["local_cum_pnl"].cummax()
    df["local_drawdown"] = df["local_cum_pnl"] - df["local_running_max"]
    df["local_rolling_20_trade_pnl"] = df["normalized_pnl_dollars"].rolling(20).sum()

    total_pnl = float(df["normalized_pnl_dollars"].sum())
    total_actual_dte = float(df["actual_dte"].sum())

    return pd.Series({
        "label": label,
        "trade_count": trade_count,
        "trade_frequency": float(trade_count / TRADE_FREQUENCY_DENOMINATOR),
        "win_rate": float(df["win"].mean()),
        "avg_pnl_per_day": float(df["pnl_per_day"].mean()),
        "median_pnl_per_day": float(df["pnl_per_day"].median()),
        "aggregate_pnl_per_day": float(total_pnl / total_actual_dte) if total_actual_dte else np.nan,
        "total_pnl": total_pnl,
        "total_actual_dte": total_actual_dte,
        "worst_trade_pnl": float(df["normalized_pnl_dollars"].min()),
        "worst_pnl_per_day": float(df["pnl_per_day"].min()),
        "max_drawdown": float(df["local_drawdown"].min()),
        "worst_20_trade_sum": float(df["local_rolling_20_trade_pnl"].min()),
        "avg_selected_tenor": float(df["selected_tenor"].mean()),
    })


def summarize_by_group(df, group_cols):
    rows = []

    for keys, sub in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {col: key for col, key in zip(group_cols, keys)}
        summary = summarize_trade_df(sub, label="group").to_dict()
        summary.pop("label", None)

        row.update(summary)
        rows.append(row)

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------
# Build qualification matrices and conditional tenor metrics
# ---------------------------------------------------------------------

core_qualifies_matrix = build_qualifies_matrix(LOCKED_2621_CORE)
secondary_qualifies_matrix = build_qualifies_matrix(LOCKED_2621_SECONDARY)

any_core_qualifies_by_date = core_qualifies_matrix.any(axis=1)
no_core_qualifies_by_date = ~any_core_qualifies_by_date

core_tenor_priority_metrics = build_layer_conditional_tenor_metrics(
    layer_name="Core",
    qualifies_matrix=core_qualifies_matrix,
    date_mask=np.ones(num_sweep_dates, dtype=bool),
)

secondary_tenor_priority_metrics = build_layer_conditional_tenor_metrics(
    layer_name="Secondary",
    qualifies_matrix=secondary_qualifies_matrix,
    date_mask=no_core_qualifies_by_date,
)

tenor_priority_metrics = pd.concat(
    [core_tenor_priority_metrics, secondary_tenor_priority_metrics],
    ignore_index=True,
)

locked_2621_threshold_table = pd.concat(
    [
        build_threshold_table(LOCKED_2621_CORE, "Core"),
        build_threshold_table(LOCKED_2621_SECONDARY, "Secondary"),
    ],
    ignore_index=True,
)

print("\nLocked 2621 qualification counts:")
print("Dates with at least one Core tenor:", int(any_core_qualifies_by_date.sum()))
print("Dates with no Core tenor:", int(no_core_qualifies_by_date.sum()))
print("Dates with at least one Secondary tenor:", int(secondary_qualifies_matrix.any(axis=1).sum()))
print(
    "Dates with at least one Secondary tenor and no Core tenor:",
    int((secondary_qualifies_matrix.any(axis=1) & no_core_qualifies_by_date).sum()),
)

print("\nCore 2621-conditional tenor priority metrics:")
display(
    core_tenor_priority_metrics.sort_values(
        ["conditional_win_probability", "conditional_avg_pnl_per_day"],
        ascending=[False, False],
    )
)

print("\nSecondary 2621-conditional tenor priority metrics:")
display(
    secondary_tenor_priority_metrics.sort_values(
        ["conditional_win_probability", "conditional_avg_pnl_per_day"],
        ascending=[False, False],
    )
)


# ---------------------------------------------------------------------
# Select final model path
# ---------------------------------------------------------------------

selected_path, selected_arrays = select_final_model_path()
locked_2621_selected_trades = attach_trade_outcomes(selected_path)

locked_2621_path_summary = pd.DataFrame(
    [summarize_trade_df(locked_2621_selected_trades, label="Combined")]
)

summary_by_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["layer"],
)

summary_by_year_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["year", "layer"],
)

summary_by_tenor_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["selected_tenor", "layer"],
)

summary_by_group_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["tenor_group", "layer"],
)

worst_trades = (
    locked_2621_selected_trades
    .sort_values("normalized_pnl_dollars", ascending=True)
    .head(20)
    .reset_index(drop=True)
)

worst_20_trade_windows = (
    locked_2621_selected_trades
    .dropna(subset=["rolling_20_trade_pnl"])
    .sort_values("rolling_20_trade_pnl", ascending=True)
    .head(20)
    .reset_index(drop=True)
)

print("\nFinal locked 2621 blended-priority path summary:")
display(locked_2621_path_summary)

print("\nSummary by layer:")
display(summary_by_layer)

print("\nSummary by year and layer:")
display(summary_by_year_layer)

print("\nSummary by tenor and layer:")
display(summary_by_tenor_layer)

print("\nSummary by tenor group and layer:")
display(summary_by_group_layer)

print("\nWorst 20 individual trades:")
display(worst_trades)

print("\nWorst 20-trade rolling windows:")
display(worst_20_trade_windows)


# ---------------------------------------------------------------------
# Save final model outputs
# ---------------------------------------------------------------------

SELECTED_TRADES_CSV_PATH = AUDIT_DIR / f"{MODEL_LABEL}_selected_trades_v0_1.csv"
SELECTED_TRADES_PARQUET_PATH = PROCESSED_DIR / f"{MODEL_LABEL}_selected_trades_v0_1.parquet"

SUMMARY_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_v0_1.csv"
SUMMARY_BY_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_layer_v0_1.csv"
SUMMARY_BY_YEAR_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_year_layer_v0_1.csv"
SUMMARY_BY_TENOR_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_tenor_layer_v0_1.csv"
SUMMARY_BY_GROUP_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_group_layer_v0_1.csv"

WORST_TRADES_PATH = AUDIT_DIR / f"{MODEL_LABEL}_worst_trades_v0_1.csv"
WORST_WINDOWS_PATH = AUDIT_DIR / f"{MODEL_LABEL}_worst_20_trade_windows_v0_1.csv"

TENOR_PRIORITY_METRICS_PATH = AUDIT_DIR / f"{MODEL_LABEL}_tenor_priority_metrics_v0_1.csv"
THRESHOLD_TABLE_PATH = AUDIT_DIR / f"{MODEL_LABEL}_thresholds_v0_1.csv"
SUMMARY_JSON_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_v0_1.json"

locked_2621_selected_trades.to_csv(SELECTED_TRADES_CSV_PATH, index=False)
locked_2621_selected_trades.to_parquet(SELECTED_TRADES_PARQUET_PATH, index=False)

locked_2621_path_summary.to_csv(SUMMARY_PATH, index=False)
summary_by_layer.to_csv(SUMMARY_BY_LAYER_PATH, index=False)
summary_by_year_layer.to_csv(SUMMARY_BY_YEAR_LAYER_PATH, index=False)
summary_by_tenor_layer.to_csv(SUMMARY_BY_TENOR_LAYER_PATH, index=False)
summary_by_group_layer.to_csv(SUMMARY_BY_GROUP_LAYER_PATH, index=False)

worst_trades.to_csv(WORST_TRADES_PATH, index=False)
worst_20_trade_windows.to_csv(WORST_WINDOWS_PATH, index=False)

tenor_priority_metrics.to_csv(TENOR_PRIORITY_METRICS_PATH, index=False)
locked_2621_threshold_table.to_csv(THRESHOLD_TABLE_PATH, index=False)

summary_json = {
    "model_label": MODEL_LABEL,
    "thresholds": "locked_2621",
    "priority_rule": "layer-specific 2621-conditional win probability within 25 bps, then conditional avg P&L/day",
    "win_band": WIN_BAND,
    "trade_count": int(locked_2621_path_summary.iloc[0]["trade_count"]),
    "trade_frequency": float(locked_2621_path_summary.iloc[0]["trade_frequency"]),
    "win_rate": float(locked_2621_path_summary.iloc[0]["win_rate"]),
    "avg_pnl_per_day": float(locked_2621_path_summary.iloc[0]["avg_pnl_per_day"]),
    "aggregate_pnl_per_day": float(locked_2621_path_summary.iloc[0]["aggregate_pnl_per_day"]),
    "total_pnl": float(locked_2621_path_summary.iloc[0]["total_pnl"]),
    "max_drawdown": float(locked_2621_path_summary.iloc[0]["max_drawdown"]),
    "worst_20_trade_sum": float(locked_2621_path_summary.iloc[0]["worst_20_trade_sum"]),
    "avg_selected_tenor": float(locked_2621_path_summary.iloc[0]["avg_selected_tenor"]),
    "selected_trades_csv": str(SELECTED_TRADES_CSV_PATH),
    "selected_trades_parquet": str(SELECTED_TRADES_PARQUET_PATH),
}

with open(SUMMARY_JSON_PATH, "w") as f:
    json.dump(summary_json, f, indent=2)

print("\nSaved final locked 2621 blended-priority outputs:")
print(SELECTED_TRADES_CSV_PATH)
print(SELECTED_TRADES_PARQUET_PATH)
print(SUMMARY_PATH)
print(SUMMARY_BY_LAYER_PATH)
print(SUMMARY_BY_YEAR_LAYER_PATH)
print(SUMMARY_BY_TENOR_LAYER_PATH)
print(SUMMARY_BY_GROUP_LAYER_PATH)
print(WORST_TRADES_PATH)
print(WORST_WINDOWS_PATH)
print(TENOR_PRIORITY_METRICS_PATH)
print(THRESHOLD_TABLE_PATH)
print(SUMMARY_JSON_PATH)

print("\nFinal locked 2621 blended-priority model complete.")

Final locked 2621 blended-priority model
Model label: locked_2621_win_band_25bps_conditional
Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
Win band: 0.0025

Locked 2621 qualification counts:
Dates with at least one Core tenor: 386
Dates with no Core tenor: 1339
Dates with at least one Secondary tenor: 577
Dates with at least one Secondary tenor and no Core tenor: 191

Core 2621-conditional tenor priority metrics:


,layer,selected_col,selected_tenor,tenor_group,conditional_win_probability,conditional_avg_pnl_per_day,conditional_median_pnl_per_day,conditional_aggregate_pnl_per_day,conditional_avg_raw_pnl,conditional_avg_actual_dte,conditional_worst_trade_pnl,conditional_worst_pnl_per_day,candidate_count
8,Core,8,33,back,0.895522,431.999112,499.797363,430.069297,"14,063.907900",32.701493,"-114,955.549452","-3,193.209707",201
6,Core,6,27,back,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208
7,Core,7,30,back,0.880000,440.960597,529.383338,440.761991,"13,156.745421",29.850000,"-57,036.901356","-2,037.032191",200
5,Core,5,24,middle,0.851240,426.816292,589.114564,430.610667,"9,886.251501",22.958678,"-57,319.352129","-2,469.960158",242
4,Core,4,21,middle,0.834783,451.823181,613.672423,450.613423,"9,811.617490",21.773913,"-56,809.083631","-2,487.128419",230
3,Core,3,18,middle,0.799213,445.583034,683.385498,451.521041,"7,805.625564",17.287402,"-45,195.769858","-2,716.007704",254
2,Core,2,15,front,0.771523,409.251409,706.225901,413.442202,"6,584.956923",15.927152,"-46,130.619927","-3,295.044280",302
0,Core,0,9,front,0.749164,459.029727,989.749243,454.055688,"3,924.013040",8.642140,"-60,891.722904","-6,576.872609",299
1,Core,1,12,front,0.745763,402.590954,843.753027,389.340289,"4,612.692582",11.847458,"-56,963.300165","-4,926.141041",295



Secondary 2621-conditional tenor priority metrics:


,layer,selected_col,selected_tenor,tenor_group,conditional_win_probability,conditional_avg_pnl_per_day,conditional_median_pnl_per_day,conditional_aggregate_pnl_per_day,conditional_avg_raw_pnl,conditional_avg_actual_dte,conditional_worst_trade_pnl,conditional_worst_pnl_per_day,candidate_count
8,Secondary,8,33,back,0.772152,184.108263,409.122134,182.709066,"6,003.958689",32.860759,"-87,932.829005","-2,512.366543",79
7,Secondary,7,30,back,0.739726,164.877502,409.122134,165.283628,"4,924.546456",29.794521,"-33,497.150782","-1,046.785962",73
2,Secondary,2,15,front,0.730435,168.629531,491.896754,164.934882,"2,650.431838",16.069565,"-42,928.530585","-2,384.918366",115
6,Secondary,6,27,back,0.710145,105.979093,395.526250,114.435826,"3,134.546532",27.391304,"-44,003.345303","-1,833.472721",69
3,Secondary,3,18,middle,0.703704,154.582398,490.623102,159.492943,"2,725.163367",17.086420,"-37,360.735722","-2,197.690337",81
5,Secondary,5,24,middle,0.703704,152.966157,444.563191,147.733099,"3,365.031706",22.777778,"-44,003.345303","-1,833.472721",81
0,Secondary,0,9,front,0.683333,107.213381,656.402509,71.258978,614.014859,8.616667,"-27,346.824178","-3,755.720924",120
4,Secondary,4,21,middle,0.662162,104.292259,448.669768,87.565238,"1,894.485762",21.635135,"-44,003.345303","-1,833.472721",74
1,Secondary,1,12,front,0.658120,-1.777573,515.605288,26.556819,304.155017,11.452991,"-29,580.217937","-3,038.536020",117



Final locked 2621 blended-priority path summary:


...


Worst 20 individual trades:


,row_idx,date,layer,selected_col,selected_tenor,tenor_group,num_qualifying_tenors_in_layer,qualifying_tenors_in_layer,selection_rule,win_band,candidate,win,normalized_pnl_dollars,actual_dte,pnl_per_day,vrp_log,vrp_z_3m,vrp_z_1y,rv21d,rsi14,spx_close,conditional_win_probability,conditional_avg_pnl_per_day,conditional_median_pnl_per_day,conditional_aggregate_pnl_per_day,conditional_avg_raw_pnl,conditional_avg_actual_dte,conditional_worst_trade_pnl,conditional_worst_pnl_per_day,candidate_count,trade_sequence,cum_pnl,running_max_cum_pnl,drawdown,rolling_20_trade_pnl,year,month
0,137,2020-01-24,Secondary,8,33,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-87,932.829005",35,"-2,512.366543",1.300539,1.061030,1.738160,7.866687,61.573549,"3,295.470000",0.772152,184.108263,409.122134,182.709066,"6,003.958689",32.860759,"-87,932.829005","-2,512.366543",79,54,"111,267.485524","199,200.314529","-87,932.829005","101,620.934570",2020,2020-01
1,157,2020-02-24,Core,0,9,front,1,9,win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-60,891.722904",11,"-5,535.611173",1.425272,0.638828,1.491332,18.033963,36.699503,"3,225.890000",0.749164,459.029727,989.749243,454.055688,"3,924.013040",8.642140,"-60,891.722904","-6,576.872609",299,60,"110,753.000943","199,200.314529","-88,447.313586","35,879.032276",2020,2020-02
2,1669,2026-03-02,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-57,319.352129",25,"-2,292.774085",1.222648,1.671398,1.140005,12.896987,48.647576,"6,881.620000",0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,557,"4,618,965.009047","4,840,074.986348","-221,109.977301","-182,593.574628",2026,2026-03
3,1668,2026-02-27,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-57,036.901356",28,"-2,037.032191",1.001127,0.847920,0.778522,12.893787,48.384049,"6,878.880000",0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,556,"4,676,284.361177","4,840,074.986348","-163,790.625172","-117,677.729809",2026,2026-02
4,1662,2026-02-19,Core,8,33,back,8,"9,12,15,18,21,24,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-52,092.062100",36,"-1,447.001725",1.045459,0.772548,0.800369,12.300219,46.444609,"6,861.890000",0.895522,431.999112,499.797363,430.069297,"14,063.907900",32.701493,"-114,955.549452","-3,193.209707",201,552,"4,787,982.924249","4,840,074.986348","-52,092.062100","32,019.152907",2026,2026-02
5,1413,2025-02-21,Core,5,24,middle,3,"9,15,24",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-49,418.522467",21,"-2,353.262975",0.741002,0.996426,0.660997,11.880124,46.090761,"6,013.130000",0.851240,426.816292,589.114564,430.610667,"9,886.251501",22.958678,"-57,319.352129","-2,469.960158",242,457,"4,164,457.294878","4,213,875.817345","-49,418.522467","150,375.347720",2025,2025-02
6,1416,2025-02-26,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-47,172.123854",30,"-1,572.404128",0.946285,1.605369,1.141095,10.753467,41.127776,"5,956.060000",0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,460,"4,046,285.657359","4,213,875.817345","-167,590.159986","-11,658.747642",2025,2025-02
7,161,2020-02-28,Core,2,15,front,3,"9,12,15",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-46,130.619927",14,"-3,295.044280",1.440955,0.832745,1.691735,24.483900,19.164013,"2,954.220000",0.771523,409.251409,706.225901,413.442202,"6,584.956923",15.927152,"-46,130.619927","-3,295.044280",302,62,"90,059.139936","199,200.314529","-109,141.174593","3,520.7


Worst 20-trade rolling windows:


,row_idx,date,layer,selected_col,selected_tenor,tenor_group,num_qualifying_tenors_in_layer,qualifying_tenors_in_layer,selection_rule,win_band,candidate,win,normalized_pnl_dollars,actual_dte,pnl_per_day,vrp_log,vrp_z_3m,vrp_z_1y,rv21d,rsi14,spx_close,conditional_win_probability,conditional_avg_pnl_per_day,conditional_median_pnl_per_day,conditional_aggregate_pnl_per_day,conditional_avg_raw_pnl,conditional_avg_actual_dte,conditional_worst_trade_pnl,conditional_worst_pnl_per_day,candidate_count,trade_sequence,cum_pnl,running_max_cum_pnl,drawdown,rolling_20_trade_pnl,year,month
0,1673,2026-03-06,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,1.000000,339.761603,27,12.583763,1.733877,3.386581,1.941130,13.851073,38.453183,"6,740.020000",0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,561,"4,565,500.729820","4,840,074.986348","-274,574.256528","-247,740.016788",2026,2026-03
1,1674,2026-03-09,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-12,685.421844",24,-528.559244,1.436394,2.039998,1.442139,13.563080,43.879944,"6,795.990000",0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,562,"4,552,815.307976","4,840,074.986348","-287,259.678373","-243,668.395666",2026,2026-03
2,1672,2026-03-05,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-17,217.829479",28,-614.922481,1.379605,2.120608,1.376651,13.201215,45.000430,"6,830.710000",0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,560,"4,565,160.968217","4,840,074.986348","-274,914.018132","-232,396.065618",2026,2026-03
3,1424,2025-03-10,Core,2,15,front,3,"9,12,15",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,1.000000,"15,449.118007",18,858.284334,0.935211,1.005079,0.791598,17.789582,29.915345,"5,614.560000",0.771523,409.251409,706.225901,413.442202,"6,584.956923",15.927152,"-46,130.619927","-3,295.044280",302,468,"3,959,245.856490","4,213,875.817345","-254,629.960854","-227,955.728566",2025,2025-03
4,1675,2026-03-10,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,1.000000,"22,148.557542",31,714.469598,1.530865,2.298511,1.583290,11.562767,42.825712,"6,781.480000",0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,563,"4,574,963.865518","4,840,074.986348","-265,111.120831","-227,324.684046",2026,2026-03
5,1423,2025-03-07,Core,2,15,front,2,"9,15",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-1,826.626460",14,-130.473319,0.903200,0.923910,0.714830,15.747961,36.833340,"5,770.200000",0.771523,409.251409,706.225901,413.442202,"6,584.956923",15.927152,"-46,130.619927","-3,295.044280",302,467,"3,943,796.738483","4,213,875.817345","-270,079.078862","-226,530.134310",2025,2025-03
6,1425,2025-03-11,Core,2,15,front,3,"9,12,15",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,1.000000,"21,051.422541",17,"1,238.318973",0.890234,0.837298,0.674164,17.724709,28.349890,"5,572.070000",0.771523,409.251409,706.225901,413.442202,"6,584.956923",15.927152,"-46,130.619927","-3,295.044280",302,469,"3,980,297.279032","4,213,875.817345","-233,578.538313","-222,433.223013",2025,2025-03
7,1671,2026-03-04,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-23,176.359269",29,-799.184802,1.164994,1.328891,1.033053,13.369281,48.264262,"6,869.500000",0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,559,"4,582,378.797696","4,840,074.986348","-257,696.188653","-217,343.93907


Saved final locked 2621 blended-priority outputs:
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_selected_trades_v0_1.csv
C:\Users\patri\vrp_project\data\processed\locked_2621_win_band_25bps_conditional_selected_trades_v0_1.parquet
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_summary_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_summary_by_layer_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_summary_by_year_layer_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_summary_by_tenor_layer_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_summary_by_group_layer_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_worst_trades_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_worst_20_trade_windows_v0_1.csv
C:\Use

## Future research roadmap

This notebook should remain the clean locked baseline. Future research should be additive and auditable.

### Next: vol-of-vol / regime overlay

The 2026 Feb–Mar weakness should be investigated as a potential regime-control problem, not as an immediate reason to overfit the tenor rule. Candidate overlay ideas to discuss/test later:

- pause new entries when vol-of-vol is elevated;
- tighten Core/Secondary thresholds in high vol-of-vol regimes;
- disable Secondary in stressed regimes;
- reduce sizing instead of changing signal eligibility;
- require stronger VRP/z-score confirmation when VVIX or vol-of-vol is rising.

### After that: sizing

Once the signal + regime overlay is settled, sizing can be layered on top. Initial sizing dimensions:

- Core vs Secondary;
- front / middle / back tenor bucket;
- vol-of-vol or stress regime;
- portfolio-level risk cap / stress-budget usage.

Avoid reintroducing exploratory sweeps or line-fitting into this baseline notebook. Put new research in a separate notebook, then bring only the chosen production candidate back here.


## Vol-of-vol overlay research add-on

This section starts the first regime-overlay test while preserving the locked baseline signal.

### Overlay concept

- Source data: Cboe VVIX daily close.
- VIX denominator: project-built 30D implied volatility from ThetaData-derived `implied_variance`, not official VIX.
- Signal: `log(VVIX / custom_vix30)`.
- Normalization: 6-month rolling z-score, initially 126 trading days.
- First tests:
  - hard filter: no new trade when z-score is above threshold;
  - size reducer: reduce P&L exposure when z-score is above threshold.

This is an overlay on the locked signal, not a replacement for the 2621 thresholds or tenor-priority rule.


In [7]:
# Cell 8 — Download and store Cboe VVIX daily history
#
# Notes:
#   - This uses daily VVIX close only.
#   - Intraday VVIX implementation is intentionally deferred.
#   - The file is stored in both raw/market and processed form so future
#     notebooks can reuse the same local copy.

import numpy as np
import pandas as pd
from pathlib import Path

VVIX_CBOE_URL = "https://cdn.cboe.com/api/global/us_indices/daily_prices/VVIX_History.csv"

RAW_MARKET_DIR = DATA_DIR / "raw" / "market"
RAW_MARKET_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_VVIX_CSV_PATH = RAW_MARKET_DIR / "vvix_history_cboe_raw.csv"
VVIX_DAILY_CSV_PATH = PROCESSED_DIR / "vvix_daily_v0_1.csv"
VVIX_DAILY_PARQUET_PATH = PROCESSED_DIR / "vvix_daily_v0_1.parquet"

print("Downloading VVIX history from Cboe...")
vvix_raw = pd.read_csv(VVIX_CBOE_URL)
vvix_raw.to_csv(RAW_VVIX_CSV_PATH, index=False)

print("Raw VVIX columns:", list(vvix_raw.columns))
print("Raw VVIX rows:", len(vvix_raw))

# Robust column mapping. Cboe has historically used DATE plus VVIX, but keep
# this tolerant in case column capitalization changes.
vvix_date_col = get_col(
    vvix_raw,
    ["DATE", "Date", "date", "trade_date", "Trade Date"],
    label="VVIX date",
)

vvix_close_col = get_col(
    vvix_raw,
    ["VVIX", "vvix", "CLOSE", "Close", "close", "PX_LAST", "Last", "last"],
    label="VVIX close",
)

vvix_daily = vvix_raw[[vvix_date_col, vvix_close_col]].copy()
vvix_daily.columns = ["date", "vvix_close"]

vvix_daily["date"] = parse_project_date_series(vvix_daily["date"]).dt.normalize()
vvix_daily["vvix_close"] = pd.to_numeric(vvix_daily["vvix_close"], errors="coerce")

vvix_daily = (
    vvix_daily
    .dropna(subset=["date", "vvix_close"])
    .loc[lambda x: x["vvix_close"] > 0]
    .drop_duplicates(subset=["date"], keep="last")
    .sort_values("date")
    .reset_index(drop=True)
)

vvix_daily.to_csv(VVIX_DAILY_CSV_PATH, index=False)
vvix_daily.to_parquet(VVIX_DAILY_PARQUET_PATH, index=False)

print("\nStored VVIX daily history:")
print("  Raw path:      ", RAW_VVIX_CSV_PATH)
print("  Processed CSV: ", VVIX_DAILY_CSV_PATH)
print("  Processed PQ:  ", VVIX_DAILY_PARQUET_PATH)
print("  Rows:          ", len(vvix_daily))
print("  Date range:    ", vvix_daily["date"].min(), "to", vvix_daily["date"].max())

print("\nVVIX daily tail:")
display(vvix_daily.tail(10))


Raw VVIX columns: ['DATE', 'VVIX']
Raw VVIX rows: 5053

Stored VVIX daily history:
  Raw path:       C:\Users\patri\vrp_project\data\raw\market\vvix_history_cboe_raw.csv
  Processed CSV:  C:\Users\patri\vrp_project\data\processed\vvix_daily_v0_1.csv
  Processed PQ:   C:\Users\patri\vrp_project\data\processed\vvix_daily_v0_1.parquet
  Rows:           5053
  Date range:     2006-03-06 00:00:00 to 2026-07-02 00:00:00

VVIX daily tail:


,date,vvix_close
5043,2026-06-18,88.430000
5044,2026-06-22,91.720000
5045,2026-06-23,99.500000
5046,2026-06-24,95.580000
5047,2026-06-25,91.190000
5048,2026-06-26,89.020000
5049,2026-06-29,88.710000
5050,2026-06-30,86.870000
5051,2026-07-01,89.040000
5052,2026-07-02,88.800000


In [8]:
# Cell 9 — Build 6-month z-score of log(VVIX / custom VIX30)
#
# Denominator:
#   custom_vix30 = sqrt(30D implied variance) * 100
#
# Why custom VIX30:
#   This keeps the overlay aligned with the project surface built from ThetaData,
#   instead of mixing in official VIX methodology.
#
# Lookahead convention:
#   By default, the rolling mean/std include the current observation.
#   This is consistent with an end-of-day signal using same-day close inputs.
#   If execution is moved earlier than the close, set
#   VOV_SHIFT_ROLLING_STATS_BY_ONE = True.

import numpy as np
import pandas as pd
from pathlib import Path

VOV_ROLLING_WINDOW = 126       # approximately 6 trading months
VOV_MIN_PERIODS = 126
VOV_SHIFT_ROLLING_STATS_BY_ONE = False

VOV_FEATURE_CSV_PATH = PROCESSED_DIR / "vol_of_vol_overlay_features_v0_1.csv"
VOV_FEATURE_PARQUET_PATH = PROCESSED_DIR / "vol_of_vol_overlay_features_v0_1.parquet"

if "features" in globals():
    features_for_vov = features.copy()
else:
    features_for_vov = pd.read_parquet(FEATURE_PANEL_PATH)
    features_for_vov["date"] = pd.to_datetime(features_for_vov["date"]).dt.normalize()

if "vvix_daily" not in globals():
    vvix_daily = pd.read_parquet(VVIX_DAILY_PARQUET_PATH)
    vvix_daily["date"] = pd.to_datetime(vvix_daily["date"]).dt.normalize()

require_columns(features_for_vov, ["date", "tenor", "implied_variance"], label="feature panel for VOV overlay")

vix30_daily = (
    features_for_vov
    .loc[features_for_vov["tenor"].astype(int).eq(30), ["date", "implied_variance"]]
    .copy()
)

vix30_daily["date"] = pd.to_datetime(vix30_daily["date"]).dt.normalize()
vix30_daily["implied_variance"] = pd.to_numeric(vix30_daily["implied_variance"], errors="coerce")
vix30_daily = vix30_daily.dropna(subset=["date", "implied_variance"])
vix30_daily = vix30_daily[vix30_daily["implied_variance"] > 0].copy()

# Most project implied-variance series are annualized decimal variance.
# In that case sqrt(var) is decimal volatility, so multiply by 100 to put it
# on the same percentage-point scale as VVIX.
vix30_daily["custom_vix30_raw_sqrt"] = np.sqrt(vix30_daily["implied_variance"])
raw_sqrt_median = float(vix30_daily["custom_vix30_raw_sqrt"].median())

if raw_sqrt_median < 3.0:
    vix_scale = 100.0
else:
    vix_scale = 1.0

vix30_daily["custom_vix30"] = vix30_daily["custom_vix30_raw_sqrt"] * vix_scale

vix30_daily = (
    vix30_daily
    .drop_duplicates(subset=["date"], keep="last")
    .sort_values("date")
    .reset_index(drop=True)
)

vov_features = (
    vix30_daily[["date", "implied_variance", "custom_vix30"]]
    .merge(vvix_daily[["date", "vvix_close"]], on="date", how="left")
    .sort_values("date")
    .reset_index(drop=True)
)

vov_features["vvix_vix_ratio"] = vov_features["vvix_close"] / vov_features["custom_vix30"]
vov_features["log_vvix_vix_ratio"] = np.log(vov_features["vvix_vix_ratio"])

ratio = vov_features["log_vvix_vix_ratio"]
rolling_mean = ratio.rolling(VOV_ROLLING_WINDOW, min_periods=VOV_MIN_PERIODS).mean()
rolling_std = ratio.rolling(VOV_ROLLING_WINDOW, min_periods=VOV_MIN_PERIODS).std(ddof=0)

if VOV_SHIFT_ROLLING_STATS_BY_ONE:
    rolling_mean = rolling_mean.shift(1)
    rolling_std = rolling_std.shift(1)

vov_features["log_vvix_vix_ratio_mean_6m"] = rolling_mean
vov_features["log_vvix_vix_ratio_std_6m"] = rolling_std

with np.errstate(divide="ignore", invalid="ignore"):
    vov_features["log_vvix_vix_ratio_z_6m"] = (
        (ratio - rolling_mean) / rolling_std
    )

vov_features.loc[
    ~np.isfinite(vov_features["log_vvix_vix_ratio_z_6m"]),
    "log_vvix_vix_ratio_z_6m",
] = np.nan

vov_features["vov_rolling_window_days"] = VOV_ROLLING_WINDOW
vov_features["vov_min_periods"] = VOV_MIN_PERIODS
vov_features["vov_shift_rolling_stats_by_one"] = VOV_SHIFT_ROLLING_STATS_BY_ONE

vov_features.to_csv(VOV_FEATURE_CSV_PATH, index=False)
vov_features.to_parquet(VOV_FEATURE_PARQUET_PATH, index=False)

print("Built vol-of-vol overlay features")
print("  VIX scale applied to sqrt(implied_variance):", vix_scale)
print("  Median raw sqrt implied variance:", raw_sqrt_median)
print("  Rows:", len(vov_features))
print("  Date range:", vov_features["date"].min(), "to", vov_features["date"].max())
print("  Non-null VVIX rows:", int(vov_features["vvix_close"].notna().sum()))
print("  Non-null z-score rows:", int(vov_features["log_vvix_vix_ratio_z_6m"].notna().sum()))
print("  Saved CSV:", VOV_FEATURE_CSV_PATH)
print("  Saved PQ: ", VOV_FEATURE_PARQUET_PATH)

print("\nOverlay feature tail:")
display(vov_features.tail(10))

print("\nOverlay feature summary:")
display(
    vov_features[
        [
            "custom_vix30",
            "vvix_close",
            "vvix_vix_ratio",
            "log_vvix_vix_ratio",
            "log_vvix_vix_ratio_z_6m",
        ]
    ].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
)


Built vol-of-vol overlay features
  VIX scale applied to sqrt(implied_variance): 100.0
  Median raw sqrt implied variance: 0.18074117643359108
  Rows: 2011
  Date range: 2018-06-25 00:00:00 to 2026-06-25 00:00:00
  Non-null VVIX rows: 2011
  Non-null z-score rows: 1886
  Saved CSV: C:\Users\patri\vrp_project\data\processed\vol_of_vol_overlay_features_v0_1.csv
  Saved PQ:  C:\Users\patri\vrp_project\data\processed\vol_of_vol_overlay_features_v0_1.parquet

Overlay feature tail:


,date,implied_variance,custom_vix30,vvix_close,vvix_vix_ratio,log_vvix_vix_ratio,log_vvix_vix_ratio_mean_6m,log_vvix_vix_ratio_std_6m,log_vvix_vix_ratio_z_6m,vov_rolling_window_days,vov_min_periods,vov_shift_rolling_stats_by_one
2001,2026-06-11,0.038205,19.546078,100.630000,5.148347,1.638676,1.695487,0.095256,-0.596407,126,126,False
2002,2026-06-12,0.031516,17.752881,93.820000,5.284776,1.664830,1.694133,0.094466,-0.310193,126,126,False
2003,2026-06-15,0.025624,16.007514,87.580000,5.471181,1.699494,1.692911,0.093387,0.070492,126,126,False
2004,2026-06-16,0.026425,16.255632,87.690000,5.394438,1.685368,1.691887,0.092754,-0.070273,126,126,False
2005,2026-06-17,0.033934,18.421125,94.530000,5.131608,1.635419,1.690578,0.092378,-0.597104,126,126,False
2006,2026-06-18,0.027484,16.578267,88.430000,5.334092,1.674119,1.689674,0.091976,-0.169121,126,126,False
2007,2026-06-22,0.029992,17.318125,91.720000,5.296185,1.666987,1.689301,0.091972,-0.242618,126,126,False
2008,2026-06-23,0.038364,19.586777,99.500000,5.079958,1.625303,1.688657,0.092133,-0.687634,126,126,False
2009,2026-06-24,0.038008,19.495519,95.580000,4.902665,1.589779,1.687642,0.092512,-1.057842,126,126,False
2010,2026-06-25,0.036027,18.980878,91.190000,4.804309,1.569513,1.686088,0.092838,-1.255679,126,126,False



Overlay feature summary:


,custom_vix30,vvix_close,vvix_vix_ratio,log_vvix_vix_ratio,log_vvix_vix_ratio_z_6m
count,"2,011.000000","2,011.000000","2,011.000000","2,011.000000","1,886.000000"
mean,19.933814,101.591462,5.423378,1.666650,0.048722
std,7.304011,16.740798,1.131969,0.226016,1.248493
min,10.611297,73.260000,2.054459,0.720012,-5.212218
1%,12.031860,76.573000,2.910040,1.068167,-3.170870
5%,12.581412,80.020000,3.412843,1.227546,-2.261031
25%,15.188156,88.830000,4.615575,1.529436,-0.738229
50%,18.074118,98.810000,5.536468,1.711357,0.152918
75%,22.706873,111.680000,6.226752,1.828855,1.010375
95%,32.003600,130.740000,7.122745,1.963293,1.808880


In [10]:
# ---------------------------------------------------------------------
# Fixed display block for second-pass vol-of-vol overlay results
# ---------------------------------------------------------------------
# Use this after the second_pass_summary dataframe has already been created.
# No need to rerun the full bakeoff.

import pandas as pd
from pathlib import Path

# If the notebook state was interrupted and second_pass_summary is not in memory,
# reload it from the saved audit file.
if "second_pass_summary" not in globals():
    SECOND_PASS_SUMMARY_PATH = Path(
        r"C:\Users\patri\vrp_project\data\audit\vol_of_vol_overlay_second_pass_bakeoff_v0_1.csv"
    )
    second_pass_summary = pd.read_csv(SECOND_PASS_SUMMARY_PATH)

# Core columns to display.
base_display_cols = [
    "signal_col",
    "scenario",
    "action",
    "threshold",
    "sample",

    "trade_count",
    "trade_frequency",
    "win_rate",
    "avg_size_multiplier",
    "notional_equiv_trade_count",

    "avg_effective_pnl_per_day",
    "aggregate_effective_pnl_per_day",
    "total_effective_pnl",
    "max_drawdown",
    "worst_20_trade_sum",

    "avg_selected_tenor",
    "avg_signal_z",
    "max_signal_z",
]

delta_display_cols = [
    "trade_count_minus_baseline",
    "trade_frequency_minus_baseline",
    "win_rate_minus_baseline",
    "notional_equiv_trade_count_minus_baseline",
    "avg_effective_pnl_per_day_minus_baseline",
    "aggregate_effective_pnl_per_day_minus_baseline",
    "total_effective_pnl_minus_baseline",
    "max_drawdown_minus_baseline",
    "worst_20_trade_sum_minus_baseline",
]

summary_display_cols = [
    c for c in base_display_cols + delta_display_cols
    if c in second_pass_summary.columns
]

def display_second_pass_candidates(
    sample,
    action,
    sort_cols,
    ascending,
    top_n=25,
):
    tmp = second_pass_summary[
        second_pass_summary["sample"].eq(sample) &
        second_pass_summary["action"].eq(action)
    ].copy()

    # Keep only sort columns that actually exist.
    valid_sort_cols = [c for c in sort_cols if c in tmp.columns]

    if valid_sort_cols:
        valid_ascending = [
            ascending[i]
            for i, c in enumerate(sort_cols)
            if c in valid_sort_cols
        ]

        tmp = tmp.sort_values(
            valid_sort_cols,
            ascending=valid_ascending,
        )

    return tmp[summary_display_cols].head(top_n)


print("\nFull-sample hard-filter candidates, sorted by max drawdown improvement:")
display(
    display_second_pass_candidates(
        sample="full_sample",
        action="no_trade_if_z_above_threshold",
        sort_cols=[
            "max_drawdown_minus_baseline",
            "worst_20_trade_sum_minus_baseline",
            "avg_effective_pnl_per_day_minus_baseline",
        ],
        ascending=[False, False, False],
        top_n=25,
    )
)

print("\n2026 Core hard-filter candidates, sorted by max drawdown improvement:")
display(
    display_second_pass_candidates(
        sample="2026_core",
        action="no_trade_if_z_above_threshold",
        sort_cols=[
            "max_drawdown_minus_baseline",
            "worst_20_trade_sum_minus_baseline",
            "total_effective_pnl_minus_baseline",
        ],
        ascending=[False, False, False],
        top_n=25,
    )
)

print("\n2026 Q1 Core hard-filter candidates, sorted by max drawdown improvement:")
display(
    display_second_pass_candidates(
        sample="2026_q1_core",
        action="no_trade_if_z_above_threshold",
        sort_cols=[
            "max_drawdown_minus_baseline",
            "worst_20_trade_sum_minus_baseline",
            "total_effective_pnl_minus_baseline",
        ],
        ascending=[False, False, False],
        top_n=25,
    )
)

print("\n2026 Feb-Mar Core hard-filter candidates, sorted by max drawdown improvement:")
display(
    display_second_pass_candidates(
        sample="2026_feb_mar_core",
        action="no_trade_if_z_above_threshold",
        sort_cols=[
            "max_drawdown_minus_baseline",
            "worst_20_trade_sum_minus_baseline",
            "total_effective_pnl_minus_baseline",
        ],
        ascending=[False, False, False],
        top_n=25,
    )
)

print("\nFull-sample 50% size-reduction candidates, sorted by max drawdown improvement:")
display(
    display_second_pass_candidates(
        sample="full_sample",
        action="reduce_size_if_z_above_threshold",
        sort_cols=[
            "max_drawdown_minus_baseline",
            "worst_20_trade_sum_minus_baseline",
            "total_effective_pnl_minus_baseline",
        ],
        ascending=[False, False, False],
        top_n=25,
    )
)

print("\n2026 Core 50% size-reduction candidates, sorted by max drawdown improvement:")
display(
    display_second_pass_candidates(
        sample="2026_core",
        action="reduce_size_if_z_above_threshold",
        sort_cols=[
            "max_drawdown_minus_baseline",
            "worst_20_trade_sum_minus_baseline",
            "total_effective_pnl_minus_baseline",
        ],
        ascending=[False, False, False],
        top_n=25,
    )
)


Full-sample hard-filter candidates, sorted by max drawdown improvement:


,signal_col,scenario,action,threshold,sample,trade_count,trade_frequency,win_rate,avg_size_multiplier,notional_equiv_trade_count,avg_effective_pnl_per_day,aggregate_effective_pnl_per_day,total_effective_pnl,max_drawdown,worst_20_trade_sum,avg_selected_tenor,avg_signal_z,max_signal_z,trade_count_minus_baseline,trade_frequency_minus_baseline,win_rate_minus_baseline,notional_equiv_trade_count_minus_baseline,avg_effective_pnl_per_day_minus_baseline,aggregate_effective_pnl_per_day_minus_baseline,total_effective_pnl_minus_baseline,max_drawdown_minus_baseline,worst_20_trade_sum_minus_baseline
7,custom_vix30_change_5d_z_6m,hard_filter__custom_vix30_change_5d_z_6m__z_gt...,no_trade_if_z_above_threshold,1.000000,full_sample,416,0.241159,0.831731,1.000000,416.000000,355.291071,341.128076,"3,096,078.417327","-221,109.977301","-196,748.485709",22.117788,0.103226,0.998707,-161.000000,-0.093333,-0.000158,-161.000000,-44.782670,-43.664636,"-1,796,560.912712","66,149.701072","50,991.531079"
161,log_custom_vix30_change_5d_z_6m,hard_filter__log_custom_vix30_change_5d_z_6m__...,no_trade_if_z_above_threshold,1.000000,full_sample,402,0.233043,0.828358,1.000000,402.000000,365.117412,360.768092,"3,175,841.511113","-233,795.399145","-205,325.066862",22.216418,0.071190,0.992132,-175.000000,-0.101449,-0.003531,-175.000000,-34.956328,-24.024620,"-1,716,797.818926","53,464.279227","42,414.949925"
623,log_vvix_z_6m,hard_filter__log_vvix_z_6m__z_gt_1.00,no_trade_if_z_above_threshold,1.000000,full_sample,401,0.232464,0.812968,1.000000,401.000000,313.456189,299.622492,"2,586,640.973886","-240,706.183910","-154,914.447616",21.830424,-0.006621,0.990056,-176.000000,-0.102029,-0.018922,-176.000000,-86.617552,-85.170220,"-2,305,998.356153","46,553.494462","92,825.569171"
637,log_vvix_z_6m,hard_filter__log_vvix_z_6m__z_gt_1.25,no_trade_if_z_above_threshold,1.250000,full_sample,438,0.253913,0.815068,1.000000,438.000000,331.681653,311.071383,"2,932,158.858506","-244,806.674560","-168,188.353785",21.842466,0.088800,1.249049,-139.000000,-0.080580,-0.016821,-139.000000,-68.392087,-73.721329,"-1,960,480.471533","42,453.003813","79,551.663003"
84,custom_vix30_change_10d_z_6m,hard_filter__custom_vix30_change_10d_z_6m__z_g...,no_trade_if_z_above_threshold,1.000000,full_sample,410,0.237681,0.831707,1.000000,410.000000,347.405209,323.451267,"2,950,845.910120","-249,162.776157","-232,062.097982",22.485366,0.196154,0.997668,-167.000000,-0.096812,-0.000182,-167.000000,-52.668532,-61.341445,"-1,941,793.419919","38,096.902216","15,677.918805"
175,log_custom_vix30_change_5d_z_6m,hard_filter__log_custom_vix30_change_5d_z_6m__...,no_trade_if_z_above_threshold,1.250000,full_sample,444,0.257391,0.819820,1.000000,444.000000,355.042725,343.130971,"3,310,870.734446","-256,971.758415","-245,409.367409",22.054054,0.169548,1.237406,-133.000000,-0.077101,-0.012069,-133.000000,-45.031016,-41.661741,"-1,581,768.595593","30,287.919958","2,330.649379"
21,custom_vix30_change_5d_z_6m,hard_filter__custom_vix30_change_5d_z_6m__z_gt...,no_trade_if_z_above_threshold,1.250000,full_sample,461,0.267246,0.822126,1.000000,461.000000,351.039135,336.121327,"3,390,791.948195","-256,971.758415","-245,409.367409",22.190889,0.202501,1.241258,-116.000000,-0.067246,-0.009763,-116.000000,-49.034605,-48.671385,"-1,501,847.381844","30,287.919958","2,330.649379"
651,log_vvix_z_6m,hard_filter__log_vvix_z_6m__z_gt_1.50,no_trade_if_z_above_threshold,1.500000,full_sample,468,0.271304,0.818376,1.000000,468.000000,341.215489,317.580082,"3,216,133.487532","-259,146.904564","-195,277.312120",21.942308,0.170445,1.484058,-109.000000,-0.063188,-0.013513,-109.000000,-58.858251,-67.212630,"-1,676,505.842507","28,112.773809","52,462.704668"
35,custom_vix30_change_5d_z_6m,hard_filter__custom_vix30_change_5d_z_6m__z_gt...,no_trade_if_z_above_threshold,1.500000,full_sample,486,0.281739,0.820988,1.000000,486.000000,359.485483,341.924169,"3,650,040.501888","-270,381.610497","-259,435.683557",22.240741,0.263384,1.499767,-91.000000,-0.0527


2026 Core hard-filter candidates, sorted by max drawdown improvement:


,signal_col,scenario,action,threshold,sample,trade_count,trade_frequency,win_rate,avg_size_multiplier,notional_equiv_trade_count,avg_effective_pnl_per_day,aggregate_effective_pnl_per_day,total_effective_pnl,max_drawdown,worst_20_trade_sum,avg_selected_tenor,avg_signal_z,max_signal_z,trade_count_minus_baseline,trade_frequency_minus_baseline,win_rate_minus_baseline,notional_equiv_trade_count_minus_baseline,avg_effective_pnl_per_day_minus_baseline,aggregate_effective_pnl_per_day_minus_baseline,total_effective_pnl_minus_baseline,max_drawdown_minus_baseline,worst_20_trade_sum_minus_baseline
627,log_vvix_z_6m,hard_filter__log_vvix_z_6m__z_gt_1.00,no_trade_if_z_above_threshold,1.000000,2026_core,16,0.009275,0.437500,1.000000,16.000000,-85.974334,-37.961627,"-15,260.574236","-66,409.805020",NaN,25.125000,0.323918,0.982561,-28.000000,-0.016232,-0.198864,-28.000000,-197.733341,-146.255433,"-138,932.100239","220,849.873353",NaN
641,log_vvix_z_6m,hard_filter__log_vvix_z_6m__z_gt_1.25,no_trade_if_z_above_threshold,1.250000,2026_core,19,0.011014,0.473684,1.000000,19.000000,-110.248893,-120.979528,"-56,497.439801","-107,646.670584",NaN,24.631579,0.442416,1.134783,-25.000000,-0.014493,-0.162679,-25.000000,-222.007899,-229.273334,"-180,168.965803","179,613.007789",NaN
655,log_vvix_z_6m,hard_filter__log_vvix_z_6m__z_gt_1.50,no_trade_if_z_above_threshold,1.500000,2026_core,25,0.014493,0.520000,1.000000,25.000000,-108.246207,-142.562338,"-89,386.586194","-213,849.423843","-184,006.457639",25.080000,0.655182,1.445355,-19.000000,-0.011014,-0.116364,-19.000000,-220.005213,-250.856144,"-213,058.112197","73,410.254530","63,733.559149"
11,custom_vix30_change_5d_z_6m,hard_filter__custom_vix30_change_5d_z_6m__z_gt...,no_trade_if_z_above_threshold,1.000000,2026_core,32,0.018551,0.625000,1.000000,32.000000,51.441108,47.600561,"38,842.057659","-221,109.977301","-196,748.485709",25.406250,0.177246,0.998707,-12.000000,-0.006957,-0.011364,-12.000000,-60.317898,-60.693245,"-84,829.468344","66,149.701072","50,991.531079"
165,log_custom_vix30_change_5d_z_6m,hard_filter__log_custom_vix30_change_5d_z_6m__...,no_trade_if_z_above_threshold,1.000000,2026_core,36,0.020870,0.666667,1.000000,36.000000,137.695998,143.975899,"132,601.803391","-233,795.399145","-205,325.066862",25.583333,0.274079,0.983110,-8.000000,-0.004638,0.030303,-8.000000,25.936991,35.682094,"8,930.277389","53,464.279227","42,414.949925"
88,custom_vix30_change_10d_z_6m,hard_filter__custom_vix30_change_10d_z_6m__z_g...,no_trade_if_z_above_threshold,1.000000,2026_core,32,0.018551,0.593750,1.000000,32.000000,22.327895,13.896055,"11,644.893935","-245,477.662635","-232,062.097982",26.062500,0.277536,0.933109,-12.000000,-0.006957,-0.042614,-12.000000,-89.431112,-94.397751,"-112,026.632067","41,782.015738","15,677.918805"
179,log_custom_vix30_change_5d_z_6m,hard_filter__log_custom_vix30_change_5d_z_6m__...,no_trade_if_z_above_threshold,1.250000,2026_core,39,0.022609,0.641026,1.000000,39.000000,126.272078,126.499437,"127,005.434293","-256,971.758415","-245,409.367409",25.692308,0.338009,1.158117,-5.000000,-0.002899,0.004662,-5.000000,14.513072,18.205631,"3,333.908291","30,287.919958","2,330.649379"
25,custom_vix30_change_5d_z_6m,hard_filter__custom_vix30_change_5d_z_6m__z_gt...,no_trade_if_z_above_threshold,1.250000,2026_core,38,0.022029,0.631579,1.000000,38.000000,105.813415,107.465922,"105,316.603431","-256,971.758415","-245,409.367409",25.657895,0.333483,1.204586,-6.000000,-0.003478,-0.004785,-6.000000,-5.945592,-0.827884,"-18,354.922571","30,287.919958","2,330.649379"
39,custom_vix30_change_5d_z_6m,hard_filter__custom_vix30_change_5d_z_6m__z_gt...,no_trade_if_z_above_threshold,1.500000,2026_core,40,0.023188,0.625000,1.000000,40.000000,111.940400,109.860331,"113,595.582211","-270,381.610497","-259,435.683557",25.725000,0.387061,1.429986,-4.000000,-0.002319,-0.011364,-4.000000,0.181393,1.566525,"-10,075.943791","16,878.067876","-11,695.666769"
193,log_custom_vix30_change_5d_z_6m,hard_filter__log_custom_vix30


2026 Q1 Core hard-filter candidates, sorted by max drawdown improvement:


...

,signal_col,scenario,action,threshold,sample,trade_count,trade_frequency,win_rate,avg_size_multiplier,notional_equiv_trade_count,avg_effective_pnl_per_day,aggregate_effective_pnl_per_day,total_effective_pnl,max_drawdown,worst_20_trade_sum,avg_selected_tenor,avg_signal_z,max_signal_z,trade_count_minus_baseline,trade_frequency_minus_baseline,win_rate_minus_baseline,notional_equiv_trade_count_minus_baseline,avg_effective_pnl_per_day_minus_baseline,aggregate_effective_pnl_per_day_minus_baseline,total_effective_pnl_minus_baseline,max_drawdown_minus_baseline,worst_20_trade_sum_minus_baseline



Full-sample 50% size-reduction candidates, sorted by max drawdown improvement:


,signal_col,scenario,action,threshold,sample,trade_count,trade_frequency,win_rate,avg_size_multiplier,notional_equiv_trade_count,avg_effective_pnl_per_day,aggregate_effective_pnl_per_day,total_effective_pnl,max_drawdown,worst_20_trade_sum,avg_selected_tenor,avg_signal_z,max_signal_z,trade_count_minus_baseline,trade_frequency_minus_baseline,win_rate_minus_baseline,notional_equiv_trade_count_minus_baseline,avg_effective_pnl_per_day_minus_baseline,aggregate_effective_pnl_per_day_minus_baseline,total_effective_pnl_minus_baseline,max_drawdown_minus_baseline,worst_20_trade_sum_minus_baseline
630,log_vvix_z_6m,reduce_50pct__log_vvix_z_6m__z_gt_1.00,reduce_size_if_z_above_threshold,1.000000,full_sample,577,0.334493,0.831889,0.847487,489.000000,308.958821,294.112478,"3,739,640.151962","-242,960.929950","-217,688.474893",22.279029,0.605063,6.215342,0.000000,0.000000,0.000000,-88.000000,-91.114919,-90.680234,"-1,152,999.178077","44,298.748423","30,051.541895"
644,log_vvix_z_6m,reduce_50pct__log_vvix_z_6m__z_gt_1.25,reduce_size_if_z_above_threshold,1.250000,full_sample,577,0.334493,0.831889,0.879549,507.500000,325.926441,307.699496,"3,912,399.094272","-252,424.469649","-233,619.948786",22.279029,0.605063,6.215342,0.000000,0.000000,0.000000,-69.500000,-74.147300,-77.093216,"-980,240.235767","34,835.208723","14,120.068002"
14,custom_vix30_change_5d_z_6m,reduce_50pct__custom_vix30_change_5d_z_6m__z_g...,reduce_size_if_z_above_threshold,1.000000,full_sample,577,0.334493,0.831889,0.860485,496.500000,328.114067,314.145409,"3,994,358.873683","-254,184.827837","-226,643.026621",22.279029,0.636003,8.452140,0.000000,0.000000,0.000000,-80.500000,-71.959673,-70.647303,"-898,280.456356","33,074.850536","21,096.990167"
658,log_vvix_z_6m,reduce_50pct__log_vvix_z_6m__z_gt_1.50,reduce_size_if_z_above_threshold,1.500000,full_sample,577,0.334493,0.831889,0.905546,522.500000,338.415422,318.866410,"4,054,386.408786","-259,594.584652","-240,790.063788",22.279029,0.605063,6.215342,0.000000,0.000000,0.000000,-54.500000,-61.658318,-65.926301,"-838,252.921253","27,665.093721","6,949.953000"
168,log_custom_vix30_change_5d_z_6m,reduce_50pct__log_custom_vix30_change_5d_z_6m_...,reduce_size_if_z_above_threshold,1.000000,full_sample,577,0.334493,0.831889,0.848354,489.500000,327.226818,317.281984,"4,034,240.420576","-260,527.538759","-224,588.606276",22.279029,0.629242,6.333223,0.000000,0.000000,0.000000,-87.500000,-72.846923,-67.510728,"-858,398.909463","26,732.139614","23,151.410512"
91,custom_vix30_change_10d_z_6m,reduce_50pct__custom_vix30_change_10d_z_6m__z_...,reduce_size_if_z_above_threshold,1.000000,full_sample,577,0.334493,0.831889,0.855286,493.500000,323.465064,308.434339,"3,921,742.620080","-265,868.391889","-244,299.832758",22.279029,0.699657,7.777289,0.000000,0.000000,0.000000,-83.500000,-76.608677,-76.358373,"-970,896.709959","21,391.286484","3,440.184030"
28,custom_vix30_change_5d_z_6m,reduce_50pct__custom_vix30_change_5d_z_6m__z_g...,reduce_size_if_z_above_threshold,1.250000,full_sample,577,0.334493,0.831889,0.899480,519.000000,340.270008,325.734616,"4,141,715.639117","-272,115.718394","-238,231.206256",22.279029,0.636003,8.452140,0.000000,0.000000,0.000000,-58.000000,-59.803732,-59.058096,"-750,923.690922","15,143.959979","9,508.810532"
182,log_custom_vix30_change_5d_z_6m,reduce_50pct__log_custom_vix30_change_5d_z_6m_...,reduce_size_if_z_above_threshold,1.250000,full_sample,577,0.334493,0.831889,0.884749,510.500000,336.639097,322.591823,"4,101,755.032242","-272,115.718394","-238,231.206256",22.279029,0.629242,6.333223,0.000000,0.000000,0.000000,-66.500000,-63.434643,-62.200889,"-790,884.297797","15,143.959979","9,508.810532"
42,custom_vix30_change_5d_z_6m,reduce_50pct__custom_vix30_change_5d_z_6m__z_g...,reduce_size_if_z_above_threshold,1.500000,full_sample,577,0.334493,0.831889,0.921144,531.500000,351.431969,335.929211,"4,271,339.915964","-278,820.644435","-244,936.132297",22.279029,0.636003,8.452140,0.000000,0.000000,0.000000,-45.500000,-48.641771,-48


2026 Core 50% size-reduction candidates, sorted by max drawdown improvement:


,signal_col,scenario,action,threshold,sample,trade_count,trade_frequency,win_rate,avg_size_multiplier,notional_equiv_trade_count,avg_effective_pnl_per_day,aggregate_effective_pnl_per_day,total_effective_pnl,max_drawdown,worst_20_trade_sum,avg_selected_tenor,avg_signal_z,max_signal_z,trade_count_minus_baseline,trade_frequency_minus_baseline,win_rate_minus_baseline,notional_equiv_trade_count_minus_baseline,avg_effective_pnl_per_day_minus_baseline,aggregate_effective_pnl_per_day_minus_baseline,total_effective_pnl_minus_baseline,max_drawdown_minus_baseline,worst_20_trade_sum_minus_baseline
634,log_vvix_z_6m,reduce_50pct__log_vvix_z_6m__z_gt_1.00,reduce_size_if_z_above_threshold,1.000000,2026_core,44,0.025507,0.636364,0.681818,30.000000,40.247806,47.465390,"54,205.475883","-167,330.393316","-154,765.903950",25.840909,1.226569,3.464874,0.000000,0.000000,0.000000,-14.000000,-71.511200,-60.828415,"-69,466.050119","119,929.285057","92,974.112837"
648,log_vvix_z_6m,reduce_50pct__log_vvix_z_6m__z_gt_1.25,reduce_size_if_z_above_threshold,1.250000,2026_core,44,0.025507,0.636364,0.715909,31.500000,32.075765,29.410721,"33,587.043101","-195,848.843994","-175,384.336733",25.840909,1.226569,3.464874,0.000000,0.000000,0.000000,-12.500000,-79.683242,-78.883085,"-90,084.482902","91,410.834379","72,355.680055"
662,log_vvix_z_6m,reduce_50pct__log_vvix_z_6m__z_gt_1.50,reduce_size_if_z_above_threshold,1.500000,2026_core,44,0.025507,0.636364,0.784091,34.500000,25.127740,15.010919,"17,142.469904","-250,554.551108","-223,760.244469",25.840909,1.226569,3.464874,0.000000,0.000000,0.000000,-9.500000,-86.631267,-93.282886,"-106,529.056098","36,705.127265","23,979.772319"
18,custom_vix30_change_5d_z_6m,reduce_50pct__custom_vix30_change_5d_z_6m__z_g...,reduce_size_if_z_above_threshold,1.000000,2026_core,44,0.025507,0.636364,0.863636,38.000000,74.585361,71.153058,"81,256.791830","-254,184.827837","-226,643.026621",25.840909,0.552937,3.360613,0.000000,0.000000,0.000000,-6.000000,-37.173646,-37.140748,"-42,414.734172","33,074.850536","21,096.990167"
172,log_custom_vix30_change_5d_z_6m,reduce_50pct__log_custom_vix30_change_5d_z_6m_...,reduce_size_if_z_above_threshold,1.000000,2026_core,44,0.025507,0.636364,0.909091,40.000000,112.209684,112.203734,"128,136.664697","-260,527.538759","-224,588.606276",25.840909,0.513332,2.674834,0.000000,0.000000,0.000000,-4.000000,0.450678,3.909929,"4,465.138694","26,732.139614","23,151.410512"
95,custom_vix30_change_10d_z_6m,reduce_50pct__custom_vix30_change_10d_z_6m__z_...,reduce_size_if_z_above_threshold,1.000000,2026_core,44,0.025507,0.636364,0.863636,38.000000,63.998738,59.245368,"67,658.209969","-265,868.391889","-244,299.832758",25.840909,0.676739,3.175118,0.000000,0.000000,0.000000,-6.000000,-47.760269,-49.048438,"-56,013.316033","21,391.286484","3,440.184030"
186,log_custom_vix30_change_5d_z_6m,reduce_50pct__log_custom_vix30_change_5d_z_6m_...,reduce_size_if_z_above_threshold,1.250000,2026_core,44,0.025507,0.636364,0.943182,41.500000,111.840993,109.753485,"125,338.480148","-272,115.718394","-238,231.206256",25.840909,0.513332,2.674834,0.000000,0.000000,0.000000,-2.500000,0.081986,1.459680,"1,666.954146","15,143.959979","9,508.810532"
32,custom_vix30_change_5d_z_6m,reduce_50pct__custom_vix30_change_5d_z_6m__z_g...,reduce_size_if_z_above_threshold,1.250000,2026_core,44,0.025507,0.636364,0.931818,41.000000,101.571660,100.257500,"114,494.064717","-272,115.718394","-238,231.206256",25.840909,0.552937,3.360613,0.000000,0.000000,0.000000,-3.000000,-10.187347,-8.036306,"-9,177.461286","15,143.959979","9,508.810532"
46,custom_vix30_change_5d_z_6m,reduce_50pct__custom_vix30_change_5d_z_6m__z_g...,reduce_size_if_z_above_threshold,1.500000,2026_core,44,0.025507,0.636364,0.954545,42.000000,106.761503,103.882272,"118,633.554107","-278,820.644435","-244,936.132297",25.840909,0.552937,3.360613,0.000000,0.000000,0.000000,-2.000000,-4.997504,-4.411534,"-5,037.971896","8,439.033938","2,803.884491"
200,log_custom_vix30_change_5d_z_6m,reduce_50

In [11]:
print(second_pass_summary["sample"].drop_duplicates().sort_values().to_list())

['2026_Feb_Mar_core', '2026_Q1_core', '2026_all', '2026_core', 'core', 'full_sample', 'secondary']


In [12]:
print("\n2026 Q1 Core hard-filter candidates, sorted by max drawdown improvement:")
display(
    display_second_pass_candidates(
        sample="2026_Q1_core",
        action="no_trade_if_z_above_threshold",
        sort_cols=[
            "max_drawdown_minus_baseline",
            "worst_20_trade_sum_minus_baseline",
            "total_effective_pnl_minus_baseline",
        ],
        ascending=[False, False, False],
        top_n=25,
    )
)

print("\n2026 Feb-Mar Core hard-filter candidates, sorted by max drawdown improvement:")
display(
    display_second_pass_candidates(
        sample="2026_Feb_Mar_core",
        action="no_trade_if_z_above_threshold",
        sort_cols=[
            "max_drawdown_minus_baseline",
            "worst_20_trade_sum_minus_baseline",
            "total_effective_pnl_minus_baseline",
        ],
        ascending=[False, False, False],
        top_n=25,
    )
)

print("\n2026 Q1 Core 50% size-reduction candidates, sorted by max drawdown improvement:")
display(
    display_second_pass_candidates(
        sample="2026_Q1_core",
        action="reduce_size_if_z_above_threshold",
        sort_cols=[
            "max_drawdown_minus_baseline",
            "worst_20_trade_sum_minus_baseline",
            "total_effective_pnl_minus_baseline",
        ],
        ascending=[False, False, False],
        top_n=25,
    )
)

print("\n2026 Feb-Mar Core 50% size-reduction candidates, sorted by max drawdown improvement:")
display(
    display_second_pass_candidates(
        sample="2026_Feb_Mar_core",
        action="reduce_size_if_z_above_threshold",
        sort_cols=[
            "max_drawdown_minus_baseline",
            "worst_20_trade_sum_minus_baseline",
            "total_effective_pnl_minus_baseline",
        ],
        ascending=[False, False, False],
        top_n=25,
    )
)


2026 Q1 Core hard-filter candidates, sorted by max drawdown improvement:


,signal_col,scenario,action,threshold,sample,trade_count,trade_frequency,win_rate,avg_size_multiplier,notional_equiv_trade_count,avg_effective_pnl_per_day,aggregate_effective_pnl_per_day,total_effective_pnl,max_drawdown,worst_20_trade_sum,avg_selected_tenor,avg_signal_z,max_signal_z,trade_count_minus_baseline,trade_frequency_minus_baseline,win_rate_minus_baseline,notional_equiv_trade_count_minus_baseline,avg_effective_pnl_per_day_minus_baseline,aggregate_effective_pnl_per_day_minus_baseline,total_effective_pnl_minus_baseline,max_drawdown_minus_baseline,worst_20_trade_sum_minus_baseline
628,log_vvix_z_6m,hard_filter__log_vvix_z_6m__z_gt_1.00,no_trade_if_z_above_threshold,1.000000,2026_Q1_core,16,0.009275,0.437500,1.000000,16.000000,-85.974334,-37.961627,"-15,260.574236","-66,409.805020",NaN,25.125000,0.323918,0.982561,-28.000000,-0.016232,-0.198864,-28.000000,-197.733341,-146.255433,"-138,932.100239","220,849.873353",NaN
642,log_vvix_z_6m,hard_filter__log_vvix_z_6m__z_gt_1.25,no_trade_if_z_above_threshold,1.250000,2026_Q1_core,19,0.011014,0.473684,1.000000,19.000000,-110.248893,-120.979528,"-56,497.439801","-107,646.670584",NaN,24.631579,0.442416,1.134783,-25.000000,-0.014493,-0.162679,-25.000000,-222.007899,-229.273334,"-180,168.965803","179,613.007789",NaN
656,log_vvix_z_6m,hard_filter__log_vvix_z_6m__z_gt_1.50,no_trade_if_z_above_threshold,1.500000,2026_Q1_core,25,0.014493,0.520000,1.000000,25.000000,-108.246207,-142.562338,"-89,386.586194","-213,849.423843","-184,006.457639",25.080000,0.655182,1.445355,-19.000000,-0.011014,-0.116364,-19.000000,-220.005213,-250.856144,"-213,058.112197","73,410.254530","63,733.559149"
12,custom_vix30_change_5d_z_6m,hard_filter__custom_vix30_change_5d_z_6m__z_gt...,no_trade_if_z_above_threshold,1.000000,2026_Q1_core,32,0.018551,0.625000,1.000000,32.000000,51.441108,47.600561,"38,842.057659","-221,109.977301","-196,748.485709",25.406250,0.177246,0.998707,-12.000000,-0.006957,-0.011364,-12.000000,-60.317898,-60.693245,"-84,829.468344","66,149.701072","50,991.531079"
166,log_custom_vix30_change_5d_z_6m,hard_filter__log_custom_vix30_change_5d_z_6m__...,no_trade_if_z_above_threshold,1.000000,2026_Q1_core,36,0.020870,0.666667,1.000000,36.000000,137.695998,143.975899,"132,601.803391","-233,795.399145","-205,325.066862",25.583333,0.274079,0.983110,-8.000000,-0.004638,0.030303,-8.000000,25.936991,35.682094,"8,930.277389","53,464.279227","42,414.949925"
89,custom_vix30_change_10d_z_6m,hard_filter__custom_vix30_change_10d_z_6m__z_g...,no_trade_if_z_above_threshold,1.000000,2026_Q1_core,32,0.018551,0.593750,1.000000,32.000000,22.327895,13.896055,"11,644.893935","-245,477.662635","-232,062.097982",26.062500,0.277536,0.933109,-12.000000,-0.006957,-0.042614,-12.000000,-89.431112,-94.397751,"-112,026.632067","41,782.015738","15,677.918805"
180,log_custom_vix30_change_5d_z_6m,hard_filter__log_custom_vix30_change_5d_z_6m__...,no_trade_if_z_above_threshold,1.250000,2026_Q1_core,39,0.022609,0.641026,1.000000,39.000000,126.272078,126.499437,"127,005.434293","-256,971.758415","-245,409.367409",25.692308,0.338009,1.158117,-5.000000,-0.002899,0.004662,-5.000000,14.513072,18.205631,"3,333.908291","30,287.919958","2,330.649379"
26,custom_vix30_change_5d_z_6m,hard_filter__custom_vix30_change_5d_z_6m__z_gt...,no_trade_if_z_above_threshold,1.250000,2026_Q1_core,38,0.022029,0.631579,1.000000,38.000000,105.813415,107.465922,"105,316.603431","-256,971.758415","-245,409.367409",25.657895,0.333483,1.204586,-6.000000,-0.003478,-0.004785,-6.000000,-5.945592,-0.827884,"-18,354.922571","30,287.919958","2,330.649379"
40,custom_vix30_change_5d_z_6m,hard_filter__custom_vix30_change_5d_z_6m__z_gt...,no_trade_if_z_above_threshold,1.500000,2026_Q1_core,40,0.023188,0.625000,1.000000,40.000000,111.940400,109.860331,"113,595.582211","-270,381.610497","-259,435.683557",25.725000,0.387061,1.429986,-4.000000,-0.002319,-0.011364,-4.000000,0.181393,1.566525,"-10,075.943791","16,878.067876","-11,695.666769"
194,log_custom_vix30_change_5d_z_6m,ha


2026 Feb-Mar Core hard-filter candidates, sorted by max drawdown improvement:


,signal_col,scenario,action,threshold,sample,trade_count,trade_frequency,win_rate,avg_size_multiplier,notional_equiv_trade_count,avg_effective_pnl_per_day,aggregate_effective_pnl_per_day,total_effective_pnl,max_drawdown,worst_20_trade_sum,avg_selected_tenor,avg_signal_z,max_signal_z,trade_count_minus_baseline,trade_frequency_minus_baseline,win_rate_minus_baseline,notional_equiv_trade_count_minus_baseline,avg_effective_pnl_per_day_minus_baseline,aggregate_effective_pnl_per_day_minus_baseline,total_effective_pnl_minus_baseline,max_drawdown_minus_baseline,worst_20_trade_sum_minus_baseline
629,log_vvix_z_6m,hard_filter__log_vvix_z_6m__z_gt_1.00,no_trade_if_z_above_threshold,1.000000,2026_Feb_Mar_core,8,0.004638,0.250000,1.000000,8.000000,-248.055499,-156.605555,"-27,092.761068","-52,223.063536",NaN,22.125000,0.702422,0.982561,-26.000000,-0.015072,-0.367647,-26.000000,-349.459549,-261.542583,"-117,443.541543","235,036.614837",NaN
643,log_vvix_z_6m,hard_filter__log_vvix_z_6m__z_gt_1.25,no_trade_if_z_above_threshold,1.250000,2026_Feb_Mar_core,10,0.005797,0.300000,1.000000,10.000000,-291.089954,-353.021298,"-74,134.472555","-104,438.009614",NaN,21.600000,0.776194,1.134783,-24.000000,-0.013913,-0.317647,-24.000000,-392.494003,-457.958325,"-164,485.253030","182,821.668758",NaN
657,log_vvix_z_6m,hard_filter__log_vvix_z_6m__z_gt_1.50,no_trade_if_z_above_threshold,1.500000,2026_Feb_Mar_core,16,0.009275,0.437500,1.000000,16.000000,-220.145359,-289.253024,"-107,023.618949","-213,849.423843",NaN,23.437500,0.983474,1.445355,-18.000000,-0.010435,-0.180147,-18.000000,-321.549408,-394.190051,"-197,374.399423","73,410.254530",NaN
13,custom_vix30_change_5d_z_6m,hard_filter__custom_vix30_change_5d_z_6m__z_gt...,no_trade_if_z_above_threshold,1.000000,2026_Feb_Mar_core,23,0.013333,0.608696,1.000000,23.000000,36.084561,37.933855,"21,205.024904","-221,109.977301","-46,936.096408",24.391304,0.112852,0.998707,-11.000000,-0.006377,-0.008951,-11.000000,-65.319489,-67.003172,"-69,145.755571","66,149.701072","180,388.587639"
167,log_custom_vix30_change_5d_z_6m,hard_filter__log_custom_vix30_change_5d_z_6m__...,no_trade_if_z_above_threshold,1.000000,2026_Feb_Mar_core,27,0.015652,0.666667,1.000000,27.000000,153.366124,173.139715,"114,964.770637","-233,795.399145","-53,014.469680",24.777778,0.229751,0.983110,-7.000000,-0.004058,0.049020,-7.000000,51.962074,68.202688,"24,613.990162","53,464.279227","174,310.214366"
90,custom_vix30_change_10d_z_6m,hard_filter__custom_vix30_change_10d_z_6m__z_g...,no_trade_if_z_above_threshold,1.000000,2026_Feb_Mar_core,23,0.013333,0.565217,1.000000,23.000000,-4.420780,-10.313492,"-5,992.138819","-244,477.105404","-81,486.541278",25.304348,0.189460,0.933109,-11.000000,-0.006377,-0.052430,-11.000000,-105.824829,-115.250519,"-96,342.919294","42,782.572968","145,838.142769"
181,log_custom_vix30_change_5d_z_6m,hard_filter__log_custom_vix30_change_5d_z_6m__...,no_trade_if_z_above_threshold,1.250000,2026_Feb_Mar_core,30,0.017391,0.633333,1.000000,30.000000,136.948016,146.410176,"109,368.401539","-256,971.758415","-120,265.295525",25.000000,0.317293,1.158117,-4.000000,-0.002319,0.015686,-4.000000,35.543966,41.473149,"19,017.621064","30,287.919958","107,059.388521"
27,custom_vix30_change_5d_z_6m,hard_filter__custom_vix30_change_5d_z_6m__z_gt...,no_trade_if_z_above_threshold,1.250000,2026_Feb_Mar_core,29,0.016812,0.620690,1.000000,29.000000,110.508248,121.271882,"87,679.570677","-256,971.758415","-120,265.295525",24.931034,0.330899,1.204586,-5.000000,-0.002899,0.003043,-5.000000,9.104198,16.334855,"-2,671.209798","30,287.919958","107,059.388521"
41,custom_vix30_change_5d_z_6m,hard_filter__custom_vix30_change_5d_z_6m__z_gt...,no_trade_if_z_above_threshold,1.500000,2026_Feb_Mar_core,31,0.017971,0.612903,1.000000,31.000000,118.111142,123.498777,"95,958.549457","-270,381.610497","-153,288.902910",25.064516,0.400199,1.429986,-3.000000,-0.001739,-0.004744,-3.000000,16.707093,18.561749,"5,607.768982","16,878.067876","74,035.781136"
195,log_custom_v


2026 Q1 Core 50% size-reduction candidates, sorted by max drawdown improvement:


,signal_col,scenario,action,threshold,sample,trade_count,trade_frequency,win_rate,avg_size_multiplier,notional_equiv_trade_count,avg_effective_pnl_per_day,aggregate_effective_pnl_per_day,total_effective_pnl,max_drawdown,worst_20_trade_sum,avg_selected_tenor,avg_signal_z,max_signal_z,trade_count_minus_baseline,trade_frequency_minus_baseline,win_rate_minus_baseline,notional_equiv_trade_count_minus_baseline,avg_effective_pnl_per_day_minus_baseline,aggregate_effective_pnl_per_day_minus_baseline,total_effective_pnl_minus_baseline,max_drawdown_minus_baseline,worst_20_trade_sum_minus_baseline
635,log_vvix_z_6m,reduce_50pct__log_vvix_z_6m__z_gt_1.00,reduce_size_if_z_above_threshold,1.000000,2026_Q1_core,44,0.025507,0.636364,0.681818,30.000000,40.247806,47.465390,"54,205.475883","-167,330.393316","-154,765.903950",25.840909,1.226569,3.464874,0.000000,0.000000,0.000000,-14.000000,-71.511200,-60.828415,"-69,466.050119","119,929.285057","92,974.112837"
649,log_vvix_z_6m,reduce_50pct__log_vvix_z_6m__z_gt_1.25,reduce_size_if_z_above_threshold,1.250000,2026_Q1_core,44,0.025507,0.636364,0.715909,31.500000,32.075765,29.410721,"33,587.043101","-195,848.843994","-175,384.336733",25.840909,1.226569,3.464874,0.000000,0.000000,0.000000,-12.500000,-79.683242,-78.883085,"-90,084.482902","91,410.834379","72,355.680055"
663,log_vvix_z_6m,reduce_50pct__log_vvix_z_6m__z_gt_1.50,reduce_size_if_z_above_threshold,1.500000,2026_Q1_core,44,0.025507,0.636364,0.784091,34.500000,25.127740,15.010919,"17,142.469904","-250,554.551108","-223,760.244469",25.840909,1.226569,3.464874,0.000000,0.000000,0.000000,-9.500000,-86.631267,-93.282886,"-106,529.056098","36,705.127265","23,979.772319"
19,custom_vix30_change_5d_z_6m,reduce_50pct__custom_vix30_change_5d_z_6m__z_g...,reduce_size_if_z_above_threshold,1.000000,2026_Q1_core,44,0.025507,0.636364,0.863636,38.000000,74.585361,71.153058,"81,256.791830","-254,184.827837","-226,643.026621",25.840909,0.552937,3.360613,0.000000,0.000000,0.000000,-6.000000,-37.173646,-37.140748,"-42,414.734172","33,074.850536","21,096.990167"
173,log_custom_vix30_change_5d_z_6m,reduce_50pct__log_custom_vix30_change_5d_z_6m_...,reduce_size_if_z_above_threshold,1.000000,2026_Q1_core,44,0.025507,0.636364,0.909091,40.000000,112.209684,112.203734,"128,136.664697","-260,527.538759","-224,588.606276",25.840909,0.513332,2.674834,0.000000,0.000000,0.000000,-4.000000,0.450678,3.909929,"4,465.138694","26,732.139614","23,151.410512"
96,custom_vix30_change_10d_z_6m,reduce_50pct__custom_vix30_change_10d_z_6m__z_...,reduce_size_if_z_above_threshold,1.000000,2026_Q1_core,44,0.025507,0.636364,0.863636,38.000000,63.998738,59.245368,"67,658.209969","-265,868.391889","-244,299.832758",25.840909,0.676739,3.175118,0.000000,0.000000,0.000000,-6.000000,-47.760269,-49.048438,"-56,013.316033","21,391.286484","3,440.184030"
187,log_custom_vix30_change_5d_z_6m,reduce_50pct__log_custom_vix30_change_5d_z_6m_...,reduce_size_if_z_above_threshold,1.250000,2026_Q1_core,44,0.025507,0.636364,0.943182,41.500000,111.840993,109.753485,"125,338.480148","-272,115.718394","-238,231.206256",25.840909,0.513332,2.674834,0.000000,0.000000,0.000000,-2.500000,0.081986,1.459680,"1,666.954146","15,143.959979","9,508.810532"
33,custom_vix30_change_5d_z_6m,reduce_50pct__custom_vix30_change_5d_z_6m__z_g...,reduce_size_if_z_above_threshold,1.250000,2026_Q1_core,44,0.025507,0.636364,0.931818,41.000000,101.571660,100.257500,"114,494.064717","-272,115.718394","-238,231.206256",25.840909,0.552937,3.360613,0.000000,0.000000,0.000000,-3.000000,-10.187347,-8.036306,"-9,177.461286","15,143.959979","9,508.810532"
47,custom_vix30_change_5d_z_6m,reduce_50pct__custom_vix30_change_5d_z_6m__z_g...,reduce_size_if_z_above_threshold,1.500000,2026_Q1_core,44,0.025507,0.636364,0.954545,42.000000,106.761503,103.882272,"118,633.554107","-278,820.644435","-244,936.132297",25.840909,0.552937,3.360613,0.000000,0.000000,0.000000,-2.000000,-4.997504,-4.411534,"-5,037.971896","8,439.033938","2,803.884491"
201,log_custom_vix


2026 Feb-Mar Core 50% size-reduction candidates, sorted by max drawdown improvement:


,signal_col,scenario,action,threshold,sample,trade_count,trade_frequency,win_rate,avg_size_multiplier,notional_equiv_trade_count,avg_effective_pnl_per_day,aggregate_effective_pnl_per_day,total_effective_pnl,max_drawdown,worst_20_trade_sum,avg_selected_tenor,avg_signal_z,max_signal_z,trade_count_minus_baseline,trade_frequency_minus_baseline,win_rate_minus_baseline,notional_equiv_trade_count_minus_baseline,avg_effective_pnl_per_day_minus_baseline,aggregate_effective_pnl_per_day_minus_baseline,total_effective_pnl_minus_baseline,max_drawdown_minus_baseline,worst_20_trade_sum_minus_baseline
636,log_vvix_z_6m,reduce_50pct__log_vvix_z_6m__z_gt_1.00,reduce_size_if_z_above_threshold,1.000000,2026_Feb_Mar_core,34,0.019710,0.617647,0.617647,21.000000,21.519025,36.735203,"31,629.009703","-167,330.393316","-136,179.716097",25.235294,1.507871,3.464874,0.000000,0.000000,0.000000,-13.000000,-79.885025,-68.201824,"-58,721.770772","119,929.285057","91,144.967950"
650,log_vvix_z_6m,reduce_50pct__log_vvix_z_6m__z_gt_1.25,reduce_size_if_z_above_threshold,1.250000,2026_Feb_Mar_core,34,0.019710,0.617647,0.647059,22.000000,7.894679,9.417136,"8,108.153960","-195,848.843994","-159,700.571840",25.235294,1.507871,3.464874,0.000000,0.000000,0.000000,-12.000000,-93.509371,-95.519891,"-82,242.626515","91,410.834379","67,624.112206"
664,log_vvix_z_6m,reduce_50pct__log_vvix_z_6m__z_gt_1.50,reduce_size_if_z_above_threshold,1.500000,2026_Feb_Mar_core,34,0.019710,0.617647,0.735294,25.000000,-1.096883,-9.682252,"-8,336.419237","-250,554.551108","-208,076.479576",25.235294,1.507871,3.464874,0.000000,0.000000,0.000000,-9.000000,-102.500933,-114.619280,"-98,687.199712","36,705.127265","19,248.204470"
20,custom_vix30_change_5d_z_6m,reduce_50pct__custom_vix30_change_5d_z_6m__z_g...,reduce_size_if_z_above_threshold,1.000000,2026_Feb_Mar_core,34,0.019710,0.617647,0.838235,28.500000,62.907097,64.782698,"55,777.902690","-254,184.827837","-199,884.982958",25.235294,0.566095,3.360613,0.000000,0.000000,0.000000,-5.500000,-38.496953,-40.154330,"-34,572.877785","33,074.850536","27,439.701089"
174,log_custom_vix30_change_5d_z_6m,reduce_50pct__log_custom_vix30_change_5d_z_6m_...,reduce_size_if_z_above_threshold,1.000000,2026_Feb_Mar_core,34,0.019710,0.617647,0.897059,30.500000,111.597398,119.230866,"102,657.775556","-260,527.538759","-204,173.273534",25.235294,0.493472,2.674834,0.000000,0.000000,0.000000,-3.500000,10.193348,14.293839,"12,306.995081","26,732.139614","23,151.410512"
97,custom_vix30_change_10d_z_6m,reduce_50pct__custom_vix30_change_10d_z_6m__z_...,reduce_size_if_z_above_threshold,1.000000,2026_Feb_Mar_core,34,0.019710,0.617647,0.838235,28.500000,49.206761,48.988758,"42,179.320828","-265,868.391889","-228,616.067865",25.235294,0.687866,3.175118,0.000000,0.000000,0.000000,-5.500000,-52.197289,-55.948269,"-48,171.459647","21,391.286484","-1,291.383819"
188,log_custom_vix30_change_5d_z_6m,reduce_50pct__log_custom_vix30_change_5d_z_6m_...,reduce_size_if_z_above_threshold,1.250000,2026_Feb_Mar_core,34,0.019710,0.617647,0.941176,32.000000,111.120267,115.980942,"99,859.591007","-272,115.718394","-217,815.873514",25.235294,0.493472,2.674834,0.000000,0.000000,0.000000,-2.000000,9.716218,11.043915,"9,508.810532","15,143.959979","9,508.810532"
34,custom_vix30_change_5d_z_6m,reduce_50pct__custom_vix30_change_5d_z_6m__z_g...,reduce_size_if_z_above_threshold,1.250000,2026_Feb_Mar_core,34,0.019710,0.617647,0.926471,31.500000,97.830542,103.385802,"89,015.175576","-272,115.718394","-217,815.873514",25.235294,0.566095,3.360613,0.000000,0.000000,0.000000,-2.500000,-3.573507,-1.551225,"-1,335.604899","15,143.959979","9,508.810532"
48,custom_vix30_change_5d_z_6m,reduce_50pct__custom_vix30_change_5d_z_6m__z_g...,reduce_size_if_z_above_threshold,1.500000,2026_Feb_Mar_core,34,0.019710,0.617647,0.955882,32.500000,104.546810,108.193571,"93,154.664966","-278,820.644435","-224,520.799555",25.235294,0.566095,3.360613,0.000000,0.000000,0.000000,-1.500000,3.142761,3.256544,"2,803.884491","8,439.0